# Extract USGS Rupture-Level Annual Occurrence Rates

This notebook loads the official USGS 2018 Conterminous United States
National Seismic Hazard Model and expands the relevant source definitions
into rupture-level annual occurrence rates.

The primary source groups considered in this notebook are:

1. Cascadia subduction-interface sources
2. Oregon intraslab sources

The resulting rupture tables will be used to generate a stochastic annual
earthquake event catalog in Notebook 3.

## Important modeling rules

- Use the official USGS model-loading and MFD-expansion implementation.
- Do not treat normalization values in `rate-tree.json` as direct annual rates.
- Keep source-scale factors separate from epistemic logic-tree weights.
- Apply each model weight exactly once.
- Preserve full-margin and partial Cascadia rupture families as additive
  source families.
- Preserve mutually exclusive logic-tree branches as alternatives.
- Retain individual rupture magnitudes, rates, geometries, and source metadata.
- Create stable rupture identifiers for reproducible catalog generation.

In [ ]:

from __future__ import annotations

from pathlib import Path
import platform
import re
import subprocess
import sys

import pandas as pd
from IPython.display import display




MODEL_NAME = "nshm-conus"
MODEL_EDITION = "2018"
MODEL_TAG = "5.2.4"

# Companion USGS calculation-code release.
#
# nshmp-haz 2.6.5 and nshm-conus 5.2.4 were both released on
# April 17, 2025, with corresponding AM09 compatibility changes.
NSHMP_HAZ_TAG = "2.6.5"
NSHMP_HAZ_COMMIT = "8015c808"



# 2. Locate the project root using the same convention as Notebook 1


CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


expected_repository_name = "seismic-correlation-insurance-loss"

if PROJECT_ROOT.name != expected_repository_name:
    raise RuntimeError(
        "The notebook does not appear to be running from the expected "
        "repository.\n\n"
        f"Expected repository name: {expected_repository_name}\n"
        f"Detected project root:     {PROJECT_ROOT}"
    )



# 3. Reconstruct the Notebook 1 model paths


RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / f"usgs_nshm_conus_{MODEL_EDITION}"
)

ARCHIVE_NAME = f"{MODEL_NAME}-{MODEL_TAG}.zip"
ARCHIVE_PATH = RAW_DATA_DIR / ARCHIVE_NAME

EXTRACT_PARENT = RAW_DATA_DIR / f"{MODEL_NAME}-{MODEL_TAG}"


def find_model_directory(extraction_directory: Path) -> Path:
    """
    Find the extracted directory containing the USGS NSHM source groups.

    The downloaded archive contains an additional nested directory, so the
    model root cannot be assumed to equal EXTRACT_PARENT directly.
    """

    if not extraction_directory.exists():
        raise FileNotFoundError(
            "The USGS model extraction directory does not exist:\n"
            f"{extraction_directory}"
        )

    possible_roots = [
        extraction_directory,
        *[
            path
            for path in extraction_directory.rglob("*")
            if path.is_dir()
        ],
    ]

    valid_candidates = []

    for candidate in possible_roots:
        required_directories_exist = all(
            (candidate / directory_name).is_dir()
            for directory_name in [
                "active-crust",
                "stable-crust",
                "subduction",
                "site-data",
            ]
        )

        if required_directories_exist:
            valid_candidates.append(candidate.resolve())

    if not valid_candidates:
        raise FileNotFoundError(
            "Could not locate a USGS model directory containing:\n"
            "  active-crust\n"
            "  stable-crust\n"
            "  subduction\n"
            "  site-data\n\n"
            f"Search directory:\n{extraction_directory}"
        )

    # Select the shallowest valid model root if more than one is found.
    valid_candidates = sorted(
        set(valid_candidates),
        key=lambda path: (
            len(path.relative_to(extraction_directory.resolve()).parts),
            str(path).lower(),
        ),
    )

    return valid_candidates[0]


MODEL_DIR = find_model_directory(EXTRACT_PARENT)



# 4. Define Notebook 2 directories

DATA_DIR = PROJECT_ROOT / "data"
METADATA_DIR = DATA_DIR / "metadata"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

TOOLS_DIR = PROJECT_ROOT / "tools"
USGS_SOFTWARE_DIR = TOOLS_DIR / "usgs_nshmp"

NSHMP_HAZ_ARCHIVE_NAME = f"nshmp-haz-{NSHMP_HAZ_TAG}.zip"

NSHMP_HAZ_ARCHIVE_PATH = (
    USGS_SOFTWARE_DIR
    / NSHMP_HAZ_ARCHIVE_NAME
)

NSHMP_HAZ_EXTRACT_PARENT = (
    USGS_SOFTWARE_DIR
    / f"nshmp-haz-{NSHMP_HAZ_TAG}"
)

JAVA_EXPORTER_DIR = (
    TOOLS_DIR
    / "usgs_rupture_rate_exporter"
)

RAW_EXPORT_DIR = (
    INTERIM_DATA_DIR
    / "usgs_rupture_rate_exports"
)

RUPTURE_RATE_DIR = (
    PROCESSED_DATA_DIR
    / "usgs_rupture_rates"
)

for directory in [
    METADATA_DIR,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    TOOLS_DIR,
    USGS_SOFTWARE_DIR,
    JAVA_EXPORTER_DIR,
    RAW_EXPORT_DIR,
    RUPTURE_RATE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )



# 5. Validate the metadata products created by Notebook 1


required_metadata_files = {
    "Complete USGS file manifest":
        METADATA_DIR / "usgs_nshm_2018_file_manifest.csv",

    "Subduction file manifest":
        METADATA_DIR / "usgs_nshm_2018_subduction_manifest.csv",

    "Cascadia geometry logic tree":
        METADATA_DIR / "cascadia_geometry_logic_tree.csv",

    "Cascadia MFD branches":
        METADATA_DIR / "cascadia_mfd_branches.csv",

    "Cascadia partial-rupture logic tree":
        METADATA_DIR / "cascadia_partial_rupture_logic_tree.csv",

    "Oregon intraslab model branches":
        METADATA_DIR / "oregon_intraslab_model_branches.csv",

    "Cascadia rupture-set inventory":
        METADATA_DIR / "cascadia_rupture_set_inventory.csv",

    "Oregon spatial PDF summary":
        METADATA_DIR / "oregon_intraslab_spatial_pdf_summary.csv",
}


metadata_validation_records = []

for description, file_path in required_metadata_files.items():

    metadata_validation_records.append(
        {
            "description": description,
            "file_name": file_path.name,
            "exists": file_path.is_file(),
            "size_kb": (
                file_path.stat().st_size / 1024
                if file_path.is_file()
                else None
            ),
        }
    )


metadata_validation = pd.DataFrame(
    metadata_validation_records
)

print("Notebook 1 metadata dependencies:")
display(metadata_validation)


missing_metadata_files = metadata_validation.loc[
    ~metadata_validation["exists"],
    "file_name",
].tolist()

if missing_metadata_files:
    raise FileNotFoundError(
        "The following Notebook 1 metadata files are missing:\n"
        + "\n".join(
            f"  - {file_name}"
            for file_name in missing_metadata_files
        )
    )



# 6. Reload the two principal Notebook 1 inventories


cascadia_rupture_set_inventory = pd.read_csv(
    required_metadata_files[
        "Cascadia rupture-set inventory"
    ]
)

oregon_spatial_pdf_summary = pd.read_csv(
    required_metadata_files[
        "Oregon spatial PDF summary"
    ]
)


inventory_validation = pd.DataFrame(
    [
        {
            "check": "Cascadia rupture-set rows",
            "expected": 21,
            "actual": len(
                cascadia_rupture_set_inventory
            ),
        },
        {
            "check": "Oregon grid points",
            "expected": 821,
            "actual": int(
                oregon_spatial_pdf_summary.loc[
                    0,
                    "number_of_grid_points",
                ]
            ),
        },
        {
            "check": "Oregon spatial PDF sum",
            "expected": 1.0,
            "actual": float(
                oregon_spatial_pdf_summary.loc[
                    0,
                    "pdf_sum",
                ]
            ),
        },
    ]
)

inventory_validation["passes"] = [
    inventory_validation.loc[
        0,
        "actual",
    ] == inventory_validation.loc[
        0,
        "expected",
    ],

    inventory_validation.loc[
        1,
        "actual",
    ] == inventory_validation.loc[
        1,
        "expected",
    ],

    abs(
        inventory_validation.loc[
            2,
            "actual",
        ]
        - inventory_validation.loc[
            2,
            "expected",
        ]
    ) < 1e-8,
]

print("\nNotebook 1 inventory checks:")
display(inventory_validation)

if not inventory_validation["passes"].all():
    raise ValueError(
        "One or more Notebook 1 inventory checks failed."
    )



# 7. Check Java and the Java compiler


def run_command(
    command: list[str],
    working_directory: Path | None = None,
) -> subprocess.CompletedProcess[str]:
    """
    Run a local command and capture its output.
    """

    return subprocess.run(
        command,
        cwd=(
            str(working_directory)
            if working_directory is not None
            else None
        ),
        capture_output=True,
        text=True,
        check=False,
    )


def combine_command_output(
    result: subprocess.CompletedProcess[str],
) -> str:
    """
    Combine stdout and stderr because Java reports its version to stderr
    on some installations.
    """

    output_parts = [
        result.stdout.strip(),
        result.stderr.strip(),
    ]

    return "\n".join(
        output
        for output in output_parts
        if output
    )


def parse_java_major_version(
    version_output: str,
) -> int | None:
    """
    Parse Java versions such as:

    java version "1.8.0_401"
    openjdk version "11.0.24"
    openjdk version "17.0.12"
    javac 21.0.4
    """

    quoted_match = re.search(
        r'version\s+"([0-9][^"]*)"',
        version_output,
        flags=re.IGNORECASE,
    )

    javac_match = re.search(
        r"\bjavac\s+([0-9][^\s]*)",
        version_output,
        flags=re.IGNORECASE,
    )

    match = quoted_match or javac_match

    if match is None:
        return None

    version_numbers = re.findall(
        r"\d+",
        match.group(1),
    )

    if not version_numbers:
        return None

    if (
        version_numbers[0] == "1"
        and len(version_numbers) >= 2
    ):
        return int(version_numbers[1])

    return int(version_numbers[0])


java_result = run_command(
    ["java", "-version"]
)

javac_result = run_command(
    ["javac", "-version"]
)

java_output = combine_command_output(
    java_result
)

javac_output = combine_command_output(
    javac_result
)

java_major_version = parse_java_major_version(
    java_output
)

javac_major_version = parse_java_major_version(
    javac_output
)

java_available = (
    java_result.returncode == 0
    and java_major_version is not None
)

javac_available = (
    javac_result.returncode == 0
    and javac_major_version is not None
)


java_validation = pd.DataFrame(
    [
        {
            "command": "java -version",
            "available": java_available,
            "major_version": java_major_version,
            "output": java_output,
        },
        {
            "command": "javac -version",
            "available": javac_available,
            "major_version": javac_major_version,
            "output": javac_output,
        },
    ]
)

print("\nJava environment:")
display(java_validation)



# 8. Validate the model directory contents


model_directory_validation = pd.DataFrame(
    [
        {
            "model_group": directory_name,
            "exists": (
                MODEL_DIR
                / directory_name
            ).is_dir(),
        }
        for directory_name in [
            "active-crust",
            "stable-crust",
            "subduction",
            "site-data",
        ]
    ]
)

print("\nUSGS model-directory validation:")
display(model_directory_validation)


subduction_file_count = sum(
    1
    for path in (
        MODEL_DIR
        / "subduction"
    ).rglob("*")
    if path.is_file()
)



# 9. Final validation report


validation_errors = []

if not model_directory_validation["exists"].all():
    validation_errors.append(
        "One or more required USGS model directories are missing."
    )

if subduction_file_count != 89:
    validation_errors.append(
        "The subduction model does not contain the 89 files verified "
        f"in Notebook 1. Found {subduction_file_count}."
    )

if not java_available:
    validation_errors.append(
        "The `java` command is unavailable."
    )

if not javac_available:
    validation_errors.append(
        "The `javac` command is unavailable. A complete Java Development "
        "Kit is required, not only a Java Runtime Environment."
    )

if (
    java_major_version is not None
    and javac_major_version is not None
    and java_major_version != javac_major_version
):
    validation_errors.append(
        "The java and javac major versions do not match. The system PATH "
        "may point to two different Java installations."
    )


print("\n" + "=" * 78)
print("NOTEBOOK 2, CELL 1: ENVIRONMENT VALIDATION")
print("=" * 78)

print(f"\nProject root:          {PROJECT_ROOT}")
print(f"USGS model directory: {MODEL_DIR}")
print(f"USGS model tag:       {MODEL_TAG}")
print(f"nshmp-haz tag:        {NSHMP_HAZ_TAG}")
print(f"nshmp-haz commit:     {NSHMP_HAZ_COMMIT}")
print(f"Subduction files:     {subduction_file_count}")
print(f"Python version:       {platform.python_version()}")
print(f"Java major version:   {java_major_version}")
print(f"javac major version:  {javac_major_version}")

print("\nNotebook 2 output directories:")
print(f"  USGS software:      {USGS_SOFTWARE_DIR}")
print(f"  Java exporter:      {JAVA_EXPORTER_DIR}")
print(f"  Raw exports:        {RAW_EXPORT_DIR}")
print(f"  Processed rates:    {RUPTURE_RATE_DIR}")


if validation_errors:

    print("\n" + "!" * 78)
    print("CELL 1 VALIDATION FAILED")
    print("!" * 78)

    for error_number, error in enumerate(
        validation_errors,
        start=1,
    ):
        print(
            f"{error_number}. {error}"
        )

    raise RuntimeError(
        "Resolve the listed environment errors before continuing "
        "to Cell 2."
    )


print("\n" + "=" * 78)
print("CELL 1 VALIDATION PASSED")
print("=" * 78)

print(
    "\nThe USGS model, Notebook 1 metadata, and Java Development Kit "
    "are available."
)

print(
    "\nNext step: download and verify the pinned nshmp-haz source release."
)

Notebook 1 metadata dependencies:


,description,file_name,exists,size_kb
0,Complete USGS file manifest,usgs_nshm_2018_file_manifest.csv,True,99.333008
1,Subduction file manifest,usgs_nshm_2018_subduction_manifest.csv,True,12.265625
2,Cascadia geometry logic tree,cascadia_geometry_logic_tree.csv,True,0.090820
3,Cascadia MFD branches,cascadia_mfd_branches.csv,True,2.591797
4,Cascadia partial-rupture logic tree,cascadia_partial_rupture_logic_tree.csv,True,0.677734
5,Oregon intraslab model branches,oregon_intraslab_model_branches.csv,True,0.564453
6,Cascadia rupture-set inventory,cascadia_rupture_set_inventory.csv,True,4.854492
7,Oregon spatial PDF summary,oregon_intraslab_spatial_pdf_summary.csv,True,0.212891



Notebook 1 inventory checks:


,check,expected,actual,passes
0,Cascadia rupture-set rows,21.0,21.0,True
1,Oregon grid points,821.0,821.0,True
2,Oregon spatial PDF sum,1.0,1.0,True



Java environment:


,command,available,major_version,output
0,java -version,True,11,"openjdk version ""11.0.32"" 2026-07-21\nOpenJDK ..."
1,javac -version,True,11,javac 11.0.32



USGS model-directory validation:


,model_group,exists
0,active-crust,True
1,stable-crust,True
2,subduction,True
3,site-data,True



NOTEBOOK 2, CELL 1: ENVIRONMENT VALIDATION

Project root:          C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss
USGS model directory: C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\raw\usgs_nshm_conus_2018\nshm-conus-5.2.4\nshm-conus-5.2.4
USGS model tag:       5.2.4
nshmp-haz tag:        2.6.5
nshmp-haz commit:     8015c808
Subduction files:     89
Python version:       3.12.3
Java major version:   11
javac major version:  11

Notebook 2 output directories:
  USGS software:      C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\tools\usgs_nshmp
  Java exporter:      C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\tools\usgs_rupture_rate_exporter
  Raw exports:        C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\interim\usgs_rupture_rate_exports
  Processed rates:    C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\usgs_rupture_rates

CELL 1 VALIDATI

In [ ]:
# ============================================================================
# NOTEBOOK 2, CELL 2
#
# Download, authenticate, verify, and extract the pinned USGS nshmp-haz
# source release.
# ============================================================================

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

import hashlib
import json
import os
import shutil
import zipfile

import pandas as pd
import requests
from IPython.display import display



# 1. Official USGS repository endpoints


NSHMP_HAZ_PROJECT_PATH = "ghsc/nshmp/nshmp-haz"

ENCODED_PROJECT_PATH = quote(
    NSHMP_HAZ_PROJECT_PATH,
    safe="",
)

ENCODED_TAG = quote(
    NSHMP_HAZ_TAG,
    safe="",
)

NSHMP_HAZ_TAG_API_URL = (
    "https://code.usgs.gov/api/v4/projects/"
    f"{ENCODED_PROJECT_PATH}/repository/tags/{ENCODED_TAG}"
)

NSHMP_HAZ_ARCHIVE_URL = (
    "https://code.usgs.gov/"
    f"{NSHMP_HAZ_PROJECT_PATH}/-/archive/"
    f"{NSHMP_HAZ_TAG}/"
    f"nshmp-haz-{NSHMP_HAZ_TAG}.zip"
)



# 2. Reproducibility metadata paths


NSHMP_HAZ_DOWNLOAD_METADATA_PATH = (
    METADATA_DIR
    / f"nshmp_haz_{NSHMP_HAZ_TAG.replace('.', '_')}_download_metadata.json"
)

NSHMP_HAZ_FILE_MANIFEST_PATH = (
    METADATA_DIR
    / f"nshmp_haz_{NSHMP_HAZ_TAG.replace('.', '_')}_file_manifest.csv"
)



# 3. Create an HTTP session


session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "seismic-correlation-insurance-loss/"
            "02_extract_usgs_rupture_rates"
        ),
        "Accept": "application/json",
    }
)



# 4. Retrieve and verify the official GitLab tag


print("=" * 78)
print("VERIFYING OFFICIAL USGS NSHMP-HAZ TAG")
print("=" * 78)

print(f"\nRequested tag:           {NSHMP_HAZ_TAG}")
print(f"Expected commit prefix:  {NSHMP_HAZ_COMMIT}")
print(f"GitLab API endpoint:     {NSHMP_HAZ_TAG_API_URL}")


try:
    tag_response = session.get(
        NSHMP_HAZ_TAG_API_URL,
        timeout=(30, 120),
    )

    tag_response.raise_for_status()

except requests.RequestException as error:
    raise RuntimeError(
        "The official USGS GitLab tag could not be retrieved.\n\n"
        f"Requested endpoint:\n{NSHMP_HAZ_TAG_API_URL}\n\n"
        f"Underlying error:\n{error}"
    ) from error


try:
    tag_metadata = tag_response.json()

except requests.JSONDecodeError as error:
    raise RuntimeError(
        "The USGS GitLab tag endpoint returned a response that could not "
        "be interpreted as JSON."
    ) from error


resolved_tag_name = str(
    tag_metadata.get(
        "name",
        "",
    )
)

commit_metadata = tag_metadata.get(
    "commit",
    {},
)

resolved_commit = str(
    commit_metadata.get(
        "id",
        "",
    )
)

resolved_short_commit = str(
    commit_metadata.get(
        "short_id",
        "",
    )
)

tag_message = tag_metadata.get(
    "message"
)

tag_release = tag_metadata.get(
    "release"
)


if resolved_tag_name != NSHMP_HAZ_TAG:
    raise RuntimeError(
        "The GitLab API returned an unexpected tag.\n\n"
        f"Expected: {NSHMP_HAZ_TAG}\n"
        f"Found:    {resolved_tag_name}"
    )


if not resolved_commit:
    raise RuntimeError(
        "The GitLab tag metadata does not contain a commit identifier."
    )


if not resolved_commit.lower().startswith(
    NSHMP_HAZ_COMMIT.lower()
):
    raise RuntimeError(
        "The USGS tag does not resolve to the expected commit.\n\n"
        f"Expected prefix: {NSHMP_HAZ_COMMIT}\n"
        f"Resolved commit: {resolved_commit}"
    )


print("\nOfficial tag verification:")
print(f"  Resolved tag:          {resolved_tag_name}")
print(f"  Resolved short commit: {resolved_short_commit}")
print(f"  Resolved full commit:  {resolved_commit}")
print("  Commit verification:   PASSED")



# 5. Utility functions


def sha256_file(
    file_path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """
    Calculate the SHA-256 checksum of a file.
    """

    digest = hashlib.sha256()

    with file_path.open("rb") as file_object:

        while True:
            chunk = file_object.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def inspect_zip_archive(
    archive_path: Path,
) -> dict:
    """
    Test a ZIP archive and return basic archive information.
    """

    if not archive_path.is_file():
        return {
            "exists": False,
            "is_zip": False,
            "is_valid": False,
            "bad_member": None,
            "member_count": 0,
            "uncompressed_size_bytes": 0,
            "top_level_entries": [],
        }

    if not zipfile.is_zipfile(
        archive_path
    ):
        return {
            "exists": True,
            "is_zip": False,
            "is_valid": False,
            "bad_member": None,
            "member_count": 0,
            "uncompressed_size_bytes": 0,
            "top_level_entries": [],
        }

    with zipfile.ZipFile(
        archive_path,
        mode="r",
    ) as zip_file:

        bad_member = zip_file.testzip()

        members = zip_file.infolist()

        top_level_entries = sorted(
            {
                Path(member.filename).parts[0]
                for member in members
                if member.filename
                and Path(member.filename).parts
            }
        )

        uncompressed_size_bytes = sum(
            member.file_size
            for member in members
        )

    return {
        "exists": True,
        "is_zip": True,
        "is_valid": bad_member is None,
        "bad_member": bad_member,
        "member_count": len(members),
        "uncompressed_size_bytes": uncompressed_size_bytes,
        "top_level_entries": top_level_entries,
    }


def download_file(
    url: str,
    output_path: Path,
    download_session: requests.Session,
    chunk_size: int = 1024 * 1024,
) -> None:
    """
    Download a file using a temporary partial file.

    The final archive path is only created after the complete response has
    been written successfully.
    """

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".part"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    try:
        response = download_session.get(
            url,
            stream=True,
            timeout=(30, 300),
            allow_redirects=True,
        )

        response.raise_for_status()

    except requests.RequestException as error:
        raise RuntimeError(
            "The nshmp-haz archive could not be downloaded.\n\n"
            f"Requested URL:\n{url}\n\n"
            f"Underlying error:\n{error}"
        ) from error

    expected_size_text = response.headers.get(
        "Content-Length"
    )

    expected_size = (
        int(expected_size_text)
        if expected_size_text
        and expected_size_text.isdigit()
        else None
    )

    downloaded_size = 0
    next_progress_report = 10 * 1024 * 1024

    try:
        with temporary_path.open(
            "wb"
        ) as output_file:

            for chunk in response.iter_content(
                chunk_size=chunk_size,
            ):

                if not chunk:
                    continue

                output_file.write(
                    chunk
                )

                downloaded_size += len(
                    chunk
                )

                if downloaded_size >= next_progress_report:

                    if expected_size:
                        percentage = (
                            downloaded_size
                            / expected_size
                            * 100
                        )

                        print(
                            "  Downloaded "
                            f"{downloaded_size / 1024**2:,.2f} MB "
                            f"of {expected_size / 1024**2:,.2f} MB "
                            f"({percentage:,.1f}%)"
                        )

                    else:
                        print(
                            "  Downloaded "
                            f"{downloaded_size / 1024**2:,.2f} MB"
                        )

                    next_progress_report += (
                        10
                        * 1024
                        * 1024
                    )

        if (
            expected_size is not None
            and downloaded_size != expected_size
        ):
            raise RuntimeError(
                "The downloaded archive size does not match the HTTP "
                "Content-Length header.\n\n"
                f"Expected bytes:   {expected_size}\n"
                f"Downloaded bytes: {downloaded_size}"
            )

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:

        if temporary_path.exists():
            temporary_path.unlink()

        raise


def safely_extract_zip(
    archive_path: Path,
    destination_directory: Path,
) -> None:
    """
    Extract a ZIP archive while preventing path-traversal entries.
    """

    destination_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    resolved_destination = (
        destination_directory.resolve()
    )

    with zipfile.ZipFile(
        archive_path,
        mode="r",
    ) as zip_file:

        for member in zip_file.infolist():

            target_path = (
                destination_directory
                / member.filename
            ).resolve()

            try:
                target_path.relative_to(
                    resolved_destination
                )

            except ValueError as error:
                raise RuntimeError(
                    "The ZIP archive contains an unsafe path:\n"
                    f"  {member.filename}"
                ) from error

        zip_file.extractall(
            destination_directory
        )


def is_nshmp_haz_source_root(
    directory: Path,
) -> bool:
    """
    Determine whether a directory is the root of the extracted Gradle
    nshmp-haz project.
    """

    required_files = [
        directory / "build.gradle",
        directory / "gradlew",
        directory / "gradlew.bat",
        directory
        / "gradle"
        / "wrapper"
        / "gradle-wrapper.properties",
        directory
        / "gradle"
        / "wrapper"
        / "gradle-wrapper.jar",
    ]

    required_directories = [
        directory
        / "src"
        / "main"
        / "java",
        directory / "gradle",
    ]

    return (
        all(
            path.is_file()
            for path in required_files
        )
        and all(
            path.is_dir()
            for path in required_directories
        )
    )


def find_nshmp_haz_source_root(
    extraction_parent: Path,
) -> Path:
    """
    Locate the extracted nshmp-haz Gradle project root.
    """

    if not extraction_parent.exists():
        raise FileNotFoundError(
            "The nshmp-haz extraction directory does not exist:\n"
            f"  {extraction_parent}"
        )

    candidates = [
        extraction_parent,
        *[
            path
            for path in extraction_parent.rglob("*")
            if path.is_dir()
        ],
    ]

    valid_candidates = [
        candidate.resolve()
        for candidate in candidates
        if is_nshmp_haz_source_root(
            candidate
        )
    ]

    valid_candidates = sorted(
        set(valid_candidates),
        key=lambda path: (
            len(
                path.relative_to(
                    extraction_parent.resolve()
                ).parts
            ),
            str(path).lower(),
        ),
    )

    if not valid_candidates:
        raise FileNotFoundError(
            "Could not locate the extracted nshmp-haz Gradle project.\n\n"
            "The expected root must contain:\n"
            "  build.gradle\n"
            "  gradlew\n"
            "  gradlew.bat\n"
            "  gradle/wrapper/gradle-wrapper.properties\n"
            "  gradle/wrapper/gradle-wrapper.jar\n"
            "  src/main/java"
        )

    return valid_candidates[0]



# 6. Validate or download the archive


print("\n" + "=" * 78)
print("DOWNLOADING AND VALIDATING NSHMP-HAZ SOURCE ARCHIVE")
print("=" * 78)

print(f"\nArchive URL:\n  {NSHMP_HAZ_ARCHIVE_URL}")
print(f"\nLocal archive:\n  {NSHMP_HAZ_ARCHIVE_PATH}")


archive_downloaded_this_run = False

existing_archive_inspection = inspect_zip_archive(
    NSHMP_HAZ_ARCHIVE_PATH
)


if (
    existing_archive_inspection["exists"]
    and existing_archive_inspection["is_valid"]
):

    print(
        "\nA valid local archive already exists. "
        "The download will not be repeated."
    )

else:

    if NSHMP_HAZ_ARCHIVE_PATH.exists():

        print(
            "\nThe existing archive is invalid and will be replaced."
        )

        NSHMP_HAZ_ARCHIVE_PATH.unlink()

    print("\nDownloading the pinned source archive...")

    download_file(
        NSHMP_HAZ_ARCHIVE_URL,
        NSHMP_HAZ_ARCHIVE_PATH,
        session,
    )

    archive_downloaded_this_run = True


archive_inspection = inspect_zip_archive(
    NSHMP_HAZ_ARCHIVE_PATH
)


if not archive_inspection["is_zip"]:
    raise RuntimeError(
        "The downloaded file is not a valid ZIP archive:\n"
        f"  {NSHMP_HAZ_ARCHIVE_PATH}"
    )


if not archive_inspection["is_valid"]:
    raise RuntimeError(
        "The ZIP integrity test failed.\n\n"
        f"First corrupted member: {archive_inspection['bad_member']}"
    )


archive_size_bytes = (
    NSHMP_HAZ_ARCHIVE_PATH.stat().st_size
)

archive_sha256 = sha256_file(
    NSHMP_HAZ_ARCHIVE_PATH
)


print("\nArchive validation:")
print(f"  Archive size:          {archive_size_bytes / 1024**2:,.2f} MB")
print(f"  Archive SHA-256:       {archive_sha256}")
print(f"  ZIP members:           {archive_inspection['member_count']:,}")
print(
    "  Uncompressed size:     "
    f"{archive_inspection['uncompressed_size_bytes'] / 1024**2:,.2f} MB"
)
print(
    "  Top-level entries:     "
    f"{archive_inspection['top_level_entries']}"
)
print("  ZIP integrity test:    PASSED")



# 7. Extract the source archive


print("\n" + "=" * 78)
print("EXTRACTING NSHMP-HAZ SOURCE")
print("=" * 78)


existing_source_root = None

if NSHMP_HAZ_EXTRACT_PARENT.exists():

    try:
        existing_source_root = find_nshmp_haz_source_root(
            NSHMP_HAZ_EXTRACT_PARENT
        )

    except FileNotFoundError:
        existing_source_root = None


if existing_source_root is not None:

    print(
        "\nA valid extracted nshmp-haz project already exists."
    )

    NSHMP_HAZ_SOURCE_DIR = existing_source_root

else:

    if NSHMP_HAZ_EXTRACT_PARENT.exists():

        print(
            "\nRemoving an incomplete or invalid previous extraction."
        )

        shutil.rmtree(
            NSHMP_HAZ_EXTRACT_PARENT
        )

    print("\nExtracting the verified source archive...")

    safely_extract_zip(
        NSHMP_HAZ_ARCHIVE_PATH,
        NSHMP_HAZ_EXTRACT_PARENT,
    )

    NSHMP_HAZ_SOURCE_DIR = find_nshmp_haz_source_root(
        NSHMP_HAZ_EXTRACT_PARENT
    )


print(f"\nLocated source root:\n  {NSHMP_HAZ_SOURCE_DIR}")



# 8. Inspect the Gradle wrapper and source tree


GRADLE_WRAPPER_PROPERTIES_PATH = (
    NSHMP_HAZ_SOURCE_DIR
    / "gradle"
    / "wrapper"
    / "gradle-wrapper.properties"
)

GRADLE_WRAPPER_JAR_PATH = (
    NSHMP_HAZ_SOURCE_DIR
    / "gradle"
    / "wrapper"
    / "gradle-wrapper.jar"
)

GRADLE_WINDOWS_LAUNCHER_PATH = (
    NSHMP_HAZ_SOURCE_DIR
    / "gradlew.bat"
)

BUILD_GRADLE_PATH = (
    NSHMP_HAZ_SOURCE_DIR
    / "build.gradle"
)

JAVA_SOURCE_DIRECTORY = (
    NSHMP_HAZ_SOURCE_DIR
    / "src"
    / "main"
    / "java"
)


gradle_wrapper_properties_text = (
    GRADLE_WRAPPER_PROPERTIES_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )
)


gradle_distribution_url = None

for line in gradle_wrapper_properties_text.splitlines():

    stripped_line = line.strip()

    if stripped_line.startswith(
        "distributionUrl="
    ):
        gradle_distribution_url = (
            stripped_line.split(
                "=",
                maxsplit=1,
            )[1]
        )
        break


java_source_file_count = sum(
    1
    for path in JAVA_SOURCE_DIRECTORY.rglob(
        "*.java"
    )
    if path.is_file()
)


source_tree_validation = pd.DataFrame(
    [
        {
            "required_item": "build.gradle",
            "path": BUILD_GRADLE_PATH,
            "exists": BUILD_GRADLE_PATH.is_file(),
        },
        {
            "required_item": "gradlew.bat",
            "path": GRADLE_WINDOWS_LAUNCHER_PATH,
            "exists": GRADLE_WINDOWS_LAUNCHER_PATH.is_file(),
        },
        {
            "required_item": "gradle-wrapper.properties",
            "path": GRADLE_WRAPPER_PROPERTIES_PATH,
            "exists": GRADLE_WRAPPER_PROPERTIES_PATH.is_file(),
        },
        {
            "required_item": "gradle-wrapper.jar",
            "path": GRADLE_WRAPPER_JAR_PATH,
            "exists": GRADLE_WRAPPER_JAR_PATH.is_file(),
        },
        {
            "required_item": "src/main/java",
            "path": JAVA_SOURCE_DIRECTORY,
            "exists": JAVA_SOURCE_DIRECTORY.is_dir(),
        },
    ]
)


print("\nGradle project validation:")
display(source_tree_validation)

print(f"\nJava source files:       {java_source_file_count:,}")
print(f"Gradle distribution:    {gradle_distribution_url}")


if not source_tree_validation["exists"].all():
    raise RuntimeError(
        "One or more required nshmp-haz build files are missing."
    )


if java_source_file_count == 0:
    raise RuntimeError(
        "No Java source files were found beneath src/main/java."
    )


if gradle_distribution_url is None:
    raise RuntimeError(
        "The Gradle distribution URL could not be read from "
        "gradle-wrapper.properties."
    )



# 9. Create a source-file manifest


source_manifest_records = []

for file_path in sorted(
    [
        path
        for path in NSHMP_HAZ_SOURCE_DIR.rglob("*")
        if path.is_file()
    ],
    key=lambda path: str(path).lower(),
):

    relative_path = file_path.relative_to(
        NSHMP_HAZ_SOURCE_DIR
    )

    source_manifest_records.append(
        {
            "relative_path": relative_path.as_posix(),
            "file_name": file_path.name,
            "suffix": file_path.suffix.lower(),
            "size_bytes": file_path.stat().st_size,
        }
    )


nshmp_haz_file_manifest = pd.DataFrame(
    source_manifest_records
)

nshmp_haz_file_manifest.to_csv(
    NSHMP_HAZ_FILE_MANIFEST_PATH,
    index=False,
)


source_file_count = len(
    nshmp_haz_file_manifest
)

source_size_bytes = int(
    nshmp_haz_file_manifest[
        "size_bytes"
    ].sum()
)


print("\nExtracted source inventory:")
print(f"  Files:                 {source_file_count:,}")
print(f"  Total size:            {source_size_bytes / 1024**2:,.2f} MB")
print(f"  Manifest:              {NSHMP_HAZ_FILE_MANIFEST_PATH}")



# 10. Save reproducibility metadata


download_metadata = {
    "software_name": "nshmp-haz",
    "official_project_path": NSHMP_HAZ_PROJECT_PATH,
    "requested_tag": NSHMP_HAZ_TAG,
    "resolved_tag": resolved_tag_name,
    "expected_commit_prefix": NSHMP_HAZ_COMMIT,
    "resolved_short_commit": resolved_short_commit,
    "resolved_full_commit": resolved_commit,
    "tag_message": tag_message,
    "tag_release": tag_release,
    "tag_api_url": NSHMP_HAZ_TAG_API_URL,
    "archive_url": NSHMP_HAZ_ARCHIVE_URL,
    "archive_path": str(
        NSHMP_HAZ_ARCHIVE_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "archive_downloaded_this_run": archive_downloaded_this_run,
    "archive_size_bytes": archive_size_bytes,
    "archive_sha256": archive_sha256,
    "zip_member_count": archive_inspection[
        "member_count"
    ],
    "zip_uncompressed_size_bytes": archive_inspection[
        "uncompressed_size_bytes"
    ],
    "zip_top_level_entries": archive_inspection[
        "top_level_entries"
    ],
    "source_directory": str(
        NSHMP_HAZ_SOURCE_DIR.relative_to(
            PROJECT_ROOT
        )
    ),
    "source_file_count": source_file_count,
    "source_size_bytes": source_size_bytes,
    "java_source_file_count": java_source_file_count,
    "gradle_distribution_url": gradle_distribution_url,
    "validated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


with NSHMP_HAZ_DOWNLOAD_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as metadata_file:

    json.dump(
        download_metadata,
        metadata_file,
        indent=2,
        ensure_ascii=False,
    )


print(
    "\nDownload metadata:\n  "
    f"{NSHMP_HAZ_DOWNLOAD_METADATA_PATH}"
)



# 11. Ensure downloaded USGS software is excluded from Git


GITIGNORE_PATH = PROJECT_ROOT / ".gitignore"

gitignore_entries = [
    "# Official USGS calculation software downloaded by Notebook 2",
    "/tools/usgs_nshmp/*.zip",
    "/tools/usgs_nshmp/nshmp-haz-*/",
    "/data/interim/usgs_rupture_rate_exports/",
    "/data/processed/usgs_rupture_rates/",
]


existing_gitignore_text = (
    GITIGNORE_PATH.read_text(
        encoding="utf-8"
    )
    if GITIGNORE_PATH.is_file()
    else ""
)

existing_gitignore_lines = set(
    existing_gitignore_text.splitlines()
)

new_gitignore_entries = [
    entry
    for entry in gitignore_entries
    if entry not in existing_gitignore_lines
]


if new_gitignore_entries:

    with GITIGNORE_PATH.open(
        "a",
        encoding="utf-8",
    ) as gitignore_file:

        if (
            existing_gitignore_text
            and not existing_gitignore_text.endswith(
                "\n"
            )
        ):
            gitignore_file.write(
                "\n"
            )

        gitignore_file.write(
            "\n"
        )

        gitignore_file.write(
            "\n".join(
                new_gitignore_entries
            )
        )

        gitignore_file.write(
            "\n"
        )


print("\n.gitignore validation:")
print(f"  File:                  {GITIGNORE_PATH}")
print(f"  New entries added:     {len(new_gitignore_entries)}")



# 12. Final Cell 2 validation


cell_2_validation = pd.DataFrame(
    [
        {
            "check": "Official tag matches requested tag",
            "passes": resolved_tag_name == NSHMP_HAZ_TAG,
        },
        {
            "check": "Official commit matches pinned commit",
            "passes": resolved_commit.lower().startswith(
                NSHMP_HAZ_COMMIT.lower()
            ),
        },
        {
            "check": "Archive exists",
            "passes": NSHMP_HAZ_ARCHIVE_PATH.is_file(),
        },
        {
            "check": "Archive passes ZIP integrity test",
            "passes": archive_inspection["is_valid"],
        },
        {
            "check": "Source directory exists",
            "passes": NSHMP_HAZ_SOURCE_DIR.is_dir(),
        },
        {
            "check": "Gradle wrapper exists",
            "passes": GRADLE_WINDOWS_LAUNCHER_PATH.is_file(),
        },
        {
            "check": "Gradle wrapper JAR exists",
            "passes": GRADLE_WRAPPER_JAR_PATH.is_file(),
        },
        {
            "check": "Java source files found",
            "passes": java_source_file_count > 0,
        },
        {
            "check": "Download metadata created",
            "passes": NSHMP_HAZ_DOWNLOAD_METADATA_PATH.is_file(),
        },
        {
            "check": "Source manifest created",
            "passes": NSHMP_HAZ_FILE_MANIFEST_PATH.is_file(),
        },
    ]
)


print("\nFinal Cell 2 validation:")
display(cell_2_validation)


if not cell_2_validation["passes"].all():

    failed_checks = cell_2_validation.loc[
        ~cell_2_validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Cell 2 validation failed:\n"
        + "\n".join(
            f"  - {check}"
            for check in failed_checks
        )
    )


print("\n" + "=" * 78)
print("CELL 2 VALIDATION PASSED")
print("=" * 78)

print(f"\nVerified nshmp-haz tag:    {resolved_tag_name}")
print(f"Verified commit:           {resolved_commit}")
print(f"Archive SHA-256:           {archive_sha256}")
print(f"Extracted source files:    {source_file_count:,}")
print(f"Java source files:         {java_source_file_count:,}")
print(f"Source directory:          {NSHMP_HAZ_SOURCE_DIR}")

print(
    "\nThe pinned official USGS nshmp-haz source release has been "
    "downloaded, authenticated, checked for corruption, extracted, "
    "and inventoried successfully."
)

print(
    "\nNext step: use the included Gradle wrapper to resolve dependencies "
    "and compile the official USGS calculation software."
)

VERIFYING OFFICIAL USGS NSHMP-HAZ TAG

Requested tag:           2.6.5
Expected commit prefix:  8015c808
GitLab API endpoint:     https://code.usgs.gov/api/v4/projects/ghsc%2Fnshmp%2Fnshmp-haz/repository/tags/2.6.5

Official tag verification:
  Resolved tag:          2.6.5
  Resolved short commit: 8015c808
  Resolved full commit:  8015c808f638fd8610fbf670227c621e6f804a43
  Commit verification:   PASSED

DOWNLOADING AND VALIDATING NSHMP-HAZ SOURCE ARCHIVE

Archive URL:
  https://code.usgs.gov/ghsc/nshmp/nshmp-haz/-/archive/2.6.5/nshmp-haz-2.6.5.zip

Local archive:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\tools\usgs_nshmp\nshmp-haz-2.6.5.zip

A valid local archive already exists. The download will not be repeated.

Archive validation:
  Archive size:          10.25 MB
  Archive SHA-256:       8d53428484b6c7d1a438f25f071ae5cec9d17638de5d24e1f504a503f61b63ff
  ZIP members:           1,055
  Uncompressed size:     11.78 MB
  Top-level entries:     ['nshmp-haz-2.6.5

,required_item,path,exists
0,build.gradle,C:\Users\USER\Documents\GitHub\seismic-correla...,True
1,gradlew.bat,C:\Users\USER\Documents\GitHub\seismic-correla...,True
2,gradle-wrapper.properties,C:\Users\USER\Documents\GitHub\seismic-correla...,True
3,gradle-wrapper.jar,C:\Users\USER\Documents\GitHub\seismic-correla...,True
4,src/main/java,C:\Users\USER\Documents\GitHub\seismic-correla...,True



Java source files:       44
Gradle distribution:    https\://services.gradle.org/distributions/gradle-7.3.1-bin.zip

Extracted source inventory:
  Files:                 790
  Total size:            11.78 MB
  Manifest:              C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_2_6_5_file_manifest.csv

Download metadata:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_2_6_5_download_metadata.json

.gitignore validation:
  File:                  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\.gitignore
  New entries added:     0

Final Cell 2 validation:


,check,passes
0,Official tag matches requested tag,True
1,Official commit matches pinned commit,True
2,Archive exists,True
3,Archive passes ZIP integrity test,True
4,Source directory exists,True
5,Gradle wrapper exists,True
6,Gradle wrapper JAR exists,True
7,Java source files found,True
8,Download metadata created,True
9,Source manifest created,True



CELL 2 VALIDATION PASSED

Verified nshmp-haz tag:    2.6.5
Verified commit:           8015c808f638fd8610fbf670227c621e6f804a43
Archive SHA-256:           8d53428484b6c7d1a438f25f071ae5cec9d17638de5d24e1f504a503f61b63ff
Extracted source files:    790
Java source files:         44
Source directory:          C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\tools\usgs_nshmp\nshmp-haz-2.6.5\nshmp-haz-2.6.5

The pinned official USGS nshmp-haz source release has been downloaded, authenticated, checked for corruption, extracted, and inventoried successfully.

Next step: use the included Gradle wrapper to resolve dependencies and compile the official USGS calculation software.


## Compile the Official USGS Calculation Software

This step uses the Gradle wrapper included with the pinned
`nshmp-haz 2.6.5` source release.

The cell:

1. Verifies the included Gradle wrapper.
2. Lists the available Gradle tasks.
3. Confirms that the standard `classes` and `jar` tasks exist.
4. Compiles the official main Java source.
5. Creates the official project JAR.
6. Inventories the compiled classes and JAR contents.
7. Records the build configuration and complete build logs.

The full test suite is not executed at this stage. The purpose of this
step is to establish a working and reproducible Java build environment
for the rupture-rate exporter.

In [7]:
# ============================================================================
# Verify the pinned Gradle wrapper and compile the official USGS nshmp-haz
# source release.
# ============================================================================

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import os
import re
import shutil
import subprocess
import zipfile

import pandas as pd
from IPython.display import display


# ----------------------------------------------------------------------------
# 1. Validate the required Cell 2 variables and paths
# ----------------------------------------------------------------------------

required_cell_2_variables = [
    "NSHMP_HAZ_SOURCE_DIR",
    "NSHMP_HAZ_TAG",
    "NSHMP_HAZ_COMMIT",
    "METADATA_DIR",
]

missing_cell_2_variables = [
    variable_name
    for variable_name in required_cell_2_variables
    if variable_name not in globals()
]

if missing_cell_2_variables:
    raise RuntimeError(
        "Cell 3 requires variables created by Cells 1 and 2.\n\n"
        "Missing variables:\n"
        + "\n".join(
            f"  - {variable_name}"
            for variable_name in missing_cell_2_variables
        )
    )


NSHMP_HAZ_SOURCE_DIR = Path(
    NSHMP_HAZ_SOURCE_DIR
).resolve()

if not NSHMP_HAZ_SOURCE_DIR.is_dir():
    raise FileNotFoundError(
        "The extracted nshmp-haz source directory does not exist:\n"
        f"  {NSHMP_HAZ_SOURCE_DIR}"
    )


# ----------------------------------------------------------------------------
# 2. Define Gradle and build-output paths
# ----------------------------------------------------------------------------

GRADLE_WRAPPER_PATH = (
    NSHMP_HAZ_SOURCE_DIR
    / "gradlew.bat"
)

GRADLE_WRAPPER_PROPERTIES_PATH = (
    NSHMP_HAZ_SOURCE_DIR
    / "gradle"
    / "wrapper"
    / "gradle-wrapper.properties"
)

BUILD_GRADLE_PATH = (
    NSHMP_HAZ_SOURCE_DIR
    / "build.gradle"
)

BUILD_DIRECTORY = (
    NSHMP_HAZ_SOURCE_DIR
    / "build"
)

COMPILED_CLASS_DIRECTORY = (
    BUILD_DIRECTORY
    / "classes"
    / "java"
    / "main"
)

JAR_OUTPUT_DIRECTORY = (
    BUILD_DIRECTORY
    / "libs"
)

BUILD_LOG_DIRECTORY = (
    METADATA_DIR
    / "nshmp_haz_build_logs"
)

BUILD_LOG_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


version_log_path = (
    BUILD_LOG_DIRECTORY
    / f"nshmp_haz_{NSHMP_HAZ_TAG.replace('.', '_')}_gradle_version.log"
)

tasks_log_path = (
    BUILD_LOG_DIRECTORY
    / f"nshmp_haz_{NSHMP_HAZ_TAG.replace('.', '_')}_gradle_tasks.log"
)

build_log_path = (
    BUILD_LOG_DIRECTORY
    / f"nshmp_haz_{NSHMP_HAZ_TAG.replace('.', '_')}_jar_build.log"
)

build_metadata_path = (
    METADATA_DIR
    / f"nshmp_haz_{NSHMP_HAZ_TAG.replace('.', '_')}_build_metadata.json"
)

jar_manifest_path = (
    METADATA_DIR
    / f"nshmp_haz_{NSHMP_HAZ_TAG.replace('.', '_')}_jar_manifest.csv"
)


# ----------------------------------------------------------------------------
# 3. Validate required Gradle project files
# ----------------------------------------------------------------------------

required_build_files = {
    "Gradle Windows wrapper":
        GRADLE_WRAPPER_PATH,

    "Gradle wrapper properties":
        GRADLE_WRAPPER_PROPERTIES_PATH,

    "Gradle build configuration":
        BUILD_GRADLE_PATH,
}


build_file_validation = pd.DataFrame(
    [
        {
            "required_item": description,
            "path": str(file_path),
            "exists": file_path.is_file(),
            "size_bytes": (
                file_path.stat().st_size
                if file_path.is_file()
                else None
            ),
        }
        for description, file_path
        in required_build_files.items()
    ]
)


print("=" * 78)
print("NOTEBOOK 2, CELL 3: COMPILE OFFICIAL NSHMP-HAZ SOFTWARE")
print("=" * 78)

print("\nRequired build files:")
display(build_file_validation)


if not build_file_validation["exists"].all():

    missing_build_files = (
        build_file_validation.loc[
            ~build_file_validation["exists"],
            "path",
        ].tolist()
    )

    raise FileNotFoundError(
        "One or more required Gradle build files are missing:\n"
        + "\n".join(
            f"  - {file_path}"
            for file_path in missing_build_files
        )
    )


# ----------------------------------------------------------------------------
# 4. Locate the Windows command processor
# ----------------------------------------------------------------------------

COMMAND_PROCESSOR = (
    os.environ.get("COMSPEC")
    or shutil.which("cmd.exe")
)


if not COMMAND_PROCESSOR:
    raise FileNotFoundError(
        "The Windows command processor cmd.exe could not be located."
    )


COMMAND_PROCESSOR = str(
    Path(COMMAND_PROCESSOR).resolve()
)


print("\nBuild environment:")
print(f"  Source directory:      {NSHMP_HAZ_SOURCE_DIR}")
print(f"  Gradle wrapper:        {GRADLE_WRAPPER_PATH}")
print(f"  Command processor:     {COMMAND_PROCESSOR}")
print(f"  Java home:             {os.environ.get('JAVA_HOME')}")
print(f"  Java major version:    {java_major_version}")
print(f"  javac major version:   {javac_major_version}")


# ----------------------------------------------------------------------------
# 5. Utility functions
# ----------------------------------------------------------------------------

def calculate_sha256(
    file_path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """
    Calculate a file's SHA-256 checksum.
    """

    digest = hashlib.sha256()

    with file_path.open("rb") as file_object:

        while True:

            chunk = file_object.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def run_gradle_command(
    gradle_arguments: list[str],
    log_path: Path,
    *,
    timeout_seconds: int = 1800,
) -> subprocess.CompletedProcess[str]:

    complete_command = [
        COMMAND_PROCESSOR,
        "/d",
        "/c",
        "call",
        str(GRADLE_WRAPPER_PATH),
        *gradle_arguments,
    ]

    environment = os.environ.copy()

    existing_gradle_opts = environment.get(
        "GRADLE_OPTS",
        "",
    ).strip()

    encoding_option = "-Dfile.encoding=UTF-8"

    if encoding_option not in existing_gradle_opts:
        environment["GRADLE_OPTS"] = (
            f"{existing_gradle_opts} {encoding_option}"
        ).strip()

    started_at = datetime.now(
        timezone.utc
    )

    try:
        result = subprocess.run(
            complete_command,
            cwd=str(NSHMP_HAZ_SOURCE_DIR),
            env=environment,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout_seconds,
            check=False,
        )

    except subprocess.TimeoutExpired as error:
        partial_output = "\n".join(
            str(value)
            for value in [
                error.stdout,
                error.stderr,
            ]
            if value
        )

        log_path.write_text(
            partial_output,
            encoding="utf-8",
        )

        raise RuntimeError(
            "The Gradle command exceeded the notebook timeout.\n\n"
            f"Partial log:\n  {log_path}"
        ) from error

    finished_at = datetime.now(
        timezone.utc
    )

    combined_output = "\n".join(
        output.strip()
        for output in [
            result.stdout,
            result.stderr,
        ]
        if output and output.strip()
    )

    displayed_command = " ".join(
        [
            f'"{GRADLE_WRAPPER_PATH}"',
            *gradle_arguments,
        ]
    )

    log_header = "\n".join(
        [
            f"Command: {displayed_command}",
            f"Working directory: {NSHMP_HAZ_SOURCE_DIR}",
            f"Started UTC: {started_at.isoformat()}",
            f"Finished UTC: {finished_at.isoformat()}",
            f"Return code: {result.returncode}",
            "",
        ]
    )

    log_path.write_text(
        log_header + combined_output + "\n",
        encoding="utf-8",
    )

    return subprocess.CompletedProcess(
        args=complete_command,
        returncode=result.returncode,
        stdout=combined_output,
        stderr="",
    )


def print_output_tail(
    output_text: str,
    maximum_lines: int = 40,
) -> None:
    """
    Print the last part of command output without flooding the notebook.
    """

    output_lines = output_text.splitlines()

    if len(output_lines) > maximum_lines:

        print(
            f"... {len(output_lines) - maximum_lines:,} earlier "
            "output lines saved to the log file ..."
        )

    for line in output_lines[
        -maximum_lines:
    ]:
        print(line)


# ----------------------------------------------------------------------------
# 6. Verify the Gradle wrapper and identify its version
# ----------------------------------------------------------------------------

print("\n" + "=" * 78)
print("VERIFYING THE PINNED GRADLE WRAPPER")
print("=" * 78)


gradle_version_result = run_gradle_command(
    [
        "--no-daemon",
        "--console=plain",
        "--version",
    ],
    version_log_path,
)


print("\nGradle wrapper output:")
print_output_tail(
    gradle_version_result.stdout,
    maximum_lines=50,
)


if gradle_version_result.returncode != 0:
    raise RuntimeError(
        "The included Gradle wrapper could not be started.\n\n"
        f"Complete log:\n  {version_log_path}"
    )


gradle_version_match = re.search(
    r"(?m)^Gradle\s+([^\s]+)",
    gradle_version_result.stdout,
)

gradle_version = (
    gradle_version_match.group(1)
    if gradle_version_match
    else None
)


if gradle_version is None:
    raise RuntimeError(
        "The Gradle wrapper ran, but its version could not be parsed.\n\n"
        f"Complete log:\n  {version_log_path}"
    )


print(f"\nDetected Gradle version: {gradle_version}")


# ----------------------------------------------------------------------------
# 7. List and verify required Gradle tasks
# ----------------------------------------------------------------------------

print("\n" + "=" * 78)
print("VERIFYING REQUIRED GRADLE TASKS")
print("=" * 78)


gradle_tasks_result = run_gradle_command(
    [
        "--no-daemon",
        "--console=plain",
        "tasks",
        "--all",
    ],
    tasks_log_path,
)


if gradle_tasks_result.returncode != 0:
    print_output_tail(
        gradle_tasks_result.stdout,
        maximum_lines=80,
    )

    raise RuntimeError(
        "The Gradle task inventory failed.\n\n"
        f"Complete log:\n  {tasks_log_path}"
    )


required_gradle_tasks = [
    "classes",
    "jar",
]


task_validation_records = []

for task_name in required_gradle_tasks:

    task_pattern = re.compile(
        rf"(?m)^\s*{re.escape(task_name)}"
        r"(?:\s+-|\s*$)"
    )

    task_validation_records.append(
        {
            "task": task_name,
            "found": bool(
                task_pattern.search(
                    gradle_tasks_result.stdout
                )
            ),
        }
    )


task_validation = pd.DataFrame(
    task_validation_records
)


print("\nRequired Gradle tasks:")
display(task_validation)


if not task_validation["found"].all():

    missing_tasks = task_validation.loc[
        ~task_validation["found"],
        "task",
    ].tolist()

    raise RuntimeError(
        "The following required Gradle tasks were not found:\n"
        + "\n".join(
            f"  - {task_name}"
            for task_name in missing_tasks
        )
        + f"\n\nComplete task inventory:\n  {tasks_log_path}"
    )


# ----------------------------------------------------------------------------
# 8. Compile the official source and create the project JAR
# ----------------------------------------------------------------------------

print("\n" + "=" * 78)
print("COMPILING OFFICIAL NSHMP-HAZ SOURCE")
print("=" * 78)

print(
    "\nRunning the pinned Gradle wrapper with:"
    "\n  clean jar --stacktrace"
)

print(
    "\nThis compiles the main Java source and creates the project JAR. "
    "The test suite is not invoked by the jar task."
)


build_started_at = datetime.now(
    timezone.utc
)

gradle_build_result = run_gradle_command(
    [
        "--no-daemon",
        "--console=plain",
        "clean",
        "jar",
        "--stacktrace",
    ],
    build_log_path,
)

build_finished_at = datetime.now(
    timezone.utc
)


print("\nGradle build output:")
print_output_tail(
    gradle_build_result.stdout,
    maximum_lines=60,
)


if gradle_build_result.returncode != 0:
    raise RuntimeError(
        "The official nshmp-haz source did not compile successfully.\n\n"
        f"Return code: {gradle_build_result.returncode}\n"
        f"Complete build log:\n  {build_log_path}"
    )


if "BUILD SUCCESSFUL" not in gradle_build_result.stdout:
    raise RuntimeError(
        "Gradle returned a zero status, but the expected "
        "'BUILD SUCCESSFUL' message was not found.\n\n"
        f"Complete build log:\n  {build_log_path}"
    )


# ----------------------------------------------------------------------------
# 9. Inventory the compiled Java classes
# ----------------------------------------------------------------------------

compiled_class_files = sorted(
    [
        path
        for path in COMPILED_CLASS_DIRECTORY.rglob(
            "*.class"
        )
        if path.is_file()
    ],
    key=lambda path: str(path).lower(),
) if COMPILED_CLASS_DIRECTORY.is_dir() else []


compiled_class_count = len(
    compiled_class_files
)


if compiled_class_count == 0:
    raise RuntimeError(
        "Gradle reported a successful build, but no compiled class files "
        "were found beneath:\n"
        f"  {COMPILED_CLASS_DIRECTORY}"
    )


compiled_class_relative_paths = [
    path.relative_to(
        COMPILED_CLASS_DIRECTORY
    ).as_posix()
    for path in compiled_class_files
]


print("\nCompiled-class inventory:")
print(f"  Class directory:       {COMPILED_CLASS_DIRECTORY}")
print(f"  Compiled class files:  {compiled_class_count:,}")


# ----------------------------------------------------------------------------
# 10. Inventory and validate the generated JAR files
# ----------------------------------------------------------------------------

generated_jar_files = sorted(
    [
        path
        for path in JAR_OUTPUT_DIRECTORY.glob(
            "*.jar"
        )
        if path.is_file()
    ],
    key=lambda path: path.name.lower(),
) if JAR_OUTPUT_DIRECTORY.is_dir() else []


if not generated_jar_files:
    raise RuntimeError(
        "Gradle reported a successful jar build, but no JAR file was "
        "found beneath:\n"
        f"  {JAR_OUTPUT_DIRECTORY}"
    )


jar_manifest_records = []

for jar_path in generated_jar_files:

    is_valid_zip = zipfile.is_zipfile(
        jar_path
    )

    jar_entry_count = 0
    jar_class_count = 0
    contains_manifest = False
    first_bad_entry = None

    if is_valid_zip:

        with zipfile.ZipFile(
            jar_path,
            mode="r",
        ) as jar_file:

            first_bad_entry = jar_file.testzip()

            jar_entries = jar_file.namelist()

            jar_entry_count = len(
                jar_entries
            )

            jar_class_count = sum(
                1
                for entry in jar_entries
                if entry.endswith(
                    ".class"
                )
            )

            contains_manifest = (
                "META-INF/MANIFEST.MF"
                in jar_entries
            )

    jar_manifest_records.append(
        {
            "jar_name": jar_path.name,
            "relative_path": jar_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
            "size_bytes": jar_path.stat().st_size,
            "size_mb": jar_path.stat().st_size / 1024**2,
            "sha256": calculate_sha256(
                jar_path
            ),
            "is_valid_zip": is_valid_zip,
            "zip_integrity_passes": (
                is_valid_zip
                and first_bad_entry is None
            ),
            "entry_count": jar_entry_count,
            "class_count": jar_class_count,
            "contains_manifest": contains_manifest,
        }
    )


jar_manifest = pd.DataFrame(
    jar_manifest_records
)

jar_manifest.to_csv(
    jar_manifest_path,
    index=False,
)


print("\nGenerated JAR inventory:")
display(jar_manifest)


if not jar_manifest["is_valid_zip"].all():
    raise RuntimeError(
        "One or more generated JAR files are not valid ZIP archives."
    )


if not jar_manifest[
    "zip_integrity_passes"
].all():
    raise RuntimeError(
        "One or more generated JAR files failed their ZIP integrity test."
    )


if jar_manifest["class_count"].sum() == 0:
    raise RuntimeError(
        "The generated JAR files do not contain compiled Java classes."
    )


# ----------------------------------------------------------------------------
# 11. Save reproducibility metadata
# ----------------------------------------------------------------------------

build_duration_seconds = (
    build_finished_at
    - build_started_at
).total_seconds()


build_metadata = {
    "software_name": "nshmp-haz",
    "software_tag": NSHMP_HAZ_TAG,
    "software_commit_prefix": NSHMP_HAZ_COMMIT,
    "resolved_full_commit": globals().get(
        "resolved_commit"
    ),
    "source_archive_sha256": globals().get(
        "archive_sha256"
    ),
    "source_directory": str(
        NSHMP_HAZ_SOURCE_DIR.relative_to(
            PROJECT_ROOT
        )
    ),
    "build_gradle_path": str(
        BUILD_GRADLE_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "gradle_wrapper_path": str(
        GRADLE_WRAPPER_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "gradle_version": gradle_version,
    "java_major_version": java_major_version,
    "javac_major_version": javac_major_version,
    "build_command": (
        "gradlew.bat --no-daemon --console=plain "
        "clean jar --stacktrace"
    ),
    "build_started_at_utc": build_started_at.isoformat(),
    "build_finished_at_utc": build_finished_at.isoformat(),
    "build_duration_seconds": build_duration_seconds,
    "build_return_code": gradle_build_result.returncode,
    "build_successful_message_found": (
        "BUILD SUCCESSFUL"
        in gradle_build_result.stdout
    ),
    "compiled_class_directory": str(
        COMPILED_CLASS_DIRECTORY.relative_to(
            PROJECT_ROOT
        )
    ),
    "compiled_class_count": compiled_class_count,
    "generated_jar_count": len(
        generated_jar_files
    ),
    "generated_jars": jar_manifest_records,
    "version_log_path": str(
        version_log_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "tasks_log_path": str(
        tasks_log_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "build_log_path": str(
        build_log_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "jar_manifest_path": str(
        jar_manifest_path.relative_to(
            PROJECT_ROOT
        )
    ),
}


with build_metadata_path.open(
    "w",
    encoding="utf-8",
) as metadata_file:

    json.dump(
        build_metadata,
        metadata_file,
        indent=2,
        ensure_ascii=False,
    )


print("\nBuild metadata:")
print(f"  {build_metadata_path}")

print("\nBuild logs:")
print(f"  Gradle version:        {version_log_path}")
print(f"  Gradle tasks:          {tasks_log_path}")
print(f"  Compilation:           {build_log_path}")

print("\nJAR manifest:")
print(f"  {jar_manifest_path}")


# ----------------------------------------------------------------------------
# 12. Final Cell 3 validation
# ----------------------------------------------------------------------------

cell_3_validation = pd.DataFrame(
    [
        {
            "check": "Gradle wrapper executed",
            "passes": (
                gradle_version_result.returncode == 0
            ),
        },
        {
            "check": "Gradle version identified",
            "passes": (
                gradle_version is not None
            ),
        },
        {
            "check": "Required Gradle tasks found",
            "passes": (
                task_validation["found"].all()
            ),
        },
        {
            "check": "Gradle jar build succeeded",
            "passes": (
                gradle_build_result.returncode == 0
                and "BUILD SUCCESSFUL"
                in gradle_build_result.stdout
            ),
        },
        {
            "check": "Compiled class files created",
            "passes": (
                compiled_class_count > 0
            ),
        },
        {
            "check": "Project JAR created",
            "passes": (
                len(generated_jar_files) > 0
            ),
        },
        {
            "check": "Generated JAR integrity passed",
            "passes": (
                jar_manifest[
                    "zip_integrity_passes"
                ].all()
            ),
        },
        {
            "check": "Generated JAR contains classes",
            "passes": (
                jar_manifest[
                    "class_count"
                ].sum() > 0
            ),
        },
        {
            "check": "Build metadata created",
            "passes": (
                build_metadata_path.is_file()
            ),
        },
        {
            "check": "JAR manifest created",
            "passes": (
                jar_manifest_path.is_file()
            ),
        },
    ]
)


print("\nFinal Cell 3 validation:")
display(cell_3_validation)


if not cell_3_validation["passes"].all():

    failed_checks = cell_3_validation.loc[
        ~cell_3_validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Cell 3 validation failed:\n"
        + "\n".join(
            f"  - {check}"
            for check in failed_checks
        )
    )


print("\n" + "=" * 78)
print("CELL 3 VALIDATION PASSED")
print("=" * 78)

print(f"\nGradle version:          {gradle_version}")
print(f"Java major version:      {java_major_version}")
print(f"Compiled class files:    {compiled_class_count:,}")
print(f"Generated JAR files:     {len(generated_jar_files):,}")
print(f"Build duration:          {build_duration_seconds:,.2f} seconds")
print(f"Build directory:         {BUILD_DIRECTORY}")

for jar_record in jar_manifest_records:

    print(
        "\nGenerated JAR:"
        f"\n  Name:                  {jar_record['jar_name']}"
        f"\n  Size:                  {jar_record['size_mb']:,.2f} MB"
        f"\n  Classes:               {jar_record['class_count']:,}"
        f"\n  SHA-256:               {jar_record['sha256']}"
    )

print(
    "\nThe pinned official USGS nshmp-haz source has been compiled "
    "successfully using its included Gradle wrapper."
)

print(
    "\nNext step: inspect the compiled USGS model APIs and create an "
    "external rupture-rate exporter without manually reconstructing "
    "the USGS magnitude-frequency distributions."
)

NOTEBOOK 2, CELL 3: COMPILE OFFICIAL NSHMP-HAZ SOFTWARE

Required build files:


,required_item,path,exists,size_bytes
0,Gradle Windows wrapper,C:\Users\USER\Documents\GitHub\seismic-correla...,True,2763
1,Gradle wrapper properties,C:\Users\USER\Documents\GitHub\seismic-correla...,True,202
2,Gradle build configuration,C:\Users\USER\Documents\GitHub\seismic-correla...,True,2705



Build environment:
  Source directory:      C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\tools\usgs_nshmp\nshmp-haz-2.6.5\nshmp-haz-2.6.5
  Gradle wrapper:        C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\tools\usgs_nshmp\nshmp-haz-2.6.5\nshmp-haz-2.6.5\gradlew.bat
  Command processor:     C:\Windows\System32\cmd.exe
  Java home:             C:\Program Files\Eclipse Adoptium\jdk-11.0.32.9-hotspot\
  Java major version:    11
  javac major version:   11

VERIFYING THE PINNED GRADLE WRAPPER

Gradle wrapper output:
------------------------------------------------------------
Gradle 7.3.1
------------------------------------------------------------

Build time:   2021-12-01 15:42:20 UTC
Revision:     2c62cec93e0b15a7d2cd68746f3348796d6d42bd

Kotlin:       1.5.31
Groovy:       3.0.9
Ant:          Apache Ant(TM) version 1.10.11 compiled on July 10 2021
JVM:          11.0.32 (Eclipse Adoptium 11.0.32+9)
OS:           Windows 11 10.0 amd64

Detecte

,task,found
0,classes,True
1,jar,True



COMPILING OFFICIAL NSHMP-HAZ SOURCE

Running the pinned Gradle wrapper with:
  clean jar --stacktrace

This compiles the main Java source and creates the project JAR. The test suite is not invoked by the jar task.

Gradle build output:
To honour the JVM settings for this build a single-use Daemon process will be forked. See https://docs.gradle.org/7.3.1/userguide/gradle_daemon.html#sec:disabling_the_daemon.
Daemon will be stopped at the end of the build 
> Task :cleanNshm UP-TO-DATE
> Task :clean
> Task :compileJava
Failed to create version file. Writing blank file.

> Task :processResources
> Task :classes
> Task :jar

Deprecated Gradle features were used in this build, making it incompatible with Gradle 8.0.

You can use '--warning-mode all' to show the individual deprecation warnings and determine if they come from your own scripts or plugins.

See https://docs.gradle.org/7.3.1/userguide/command_line_interface.html#sec:command_line_warnings

BUILD SUCCESSFUL in 2m 9s
5 actionable t

,jar_name,relative_path,size_bytes,size_mb,sha256,is_valid_zip,zip_integrity_passes,entry_count,class_count,contains_manifest
0,nshmp-haz-thin.jar,tools/usgs_nshmp/nshmp-haz-2.6.5/nshmp-haz-2.6...,423433,0.403817,93822a39b8eb87f85e52d443de3bd60b8ec49b5fa42739...,True,True,302,235,True



Build metadata:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_2_6_5_build_metadata.json

Build logs:
  Gradle version:        C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_build_logs\nshmp_haz_2_6_5_gradle_version.log
  Gradle tasks:          C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_build_logs\nshmp_haz_2_6_5_gradle_tasks.log
  Compilation:           C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_build_logs\nshmp_haz_2_6_5_jar_build.log

JAR manifest:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_2_6_5_jar_manifest.csv

Final Cell 3 validation:


,check,passes
0,Gradle wrapper executed,True
1,Gradle version identified,True
2,Required Gradle tasks found,True
3,Gradle jar build succeeded,True
4,Compiled class files created,True
5,Project JAR created,True
6,Generated JAR integrity passed,True
7,Generated JAR contains classes,True
8,Build metadata created,True
9,JAR manifest created,True



CELL 3 VALIDATION PASSED

Gradle version:          7.3.1
Java major version:      11
Compiled class files:    235
Generated JAR files:     1
Build duration:          131.14 seconds
Build directory:         C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\tools\usgs_nshmp\nshmp-haz-2.6.5\nshmp-haz-2.6.5\build

Generated JAR:
  Name:                  nshmp-haz-thin.jar
  Size:                  0.40 MB
  Classes:               235
  SHA-256:               93822a39b8eb87f85e52d443de3bd60b8ec49b5fa42739a2edda683d6889b1b0

The pinned official USGS nshmp-haz source has been compiled successfully using its included Gradle wrapper.

Next step: inspect the compiled USGS model APIs and create an external rupture-rate exporter without manually reconstructing the USGS magnitude-frequency distributions.


In [ ]:
# from pathlib import Path
# import os
# import shutil
# import subprocess

# log_path = Path(
#     r"."
#     r"\data\metadata\nshmp_haz_build_logs"
#     r"\nshmp_haz_2_6_5_gradle_version.log"
# )

# print("Saved Gradle log")
# print("=" * 60)

# if log_path.is_file():
#     print(
#         log_path.read_text(
#             encoding="utf-8",
#             errors="replace",
#         )
#     )
# else:
#     print(f"Log file not found:\n{log_path}")


# source_dir = Path(NSHMP_HAZ_SOURCE_DIR).resolve()
# wrapper_path = source_dir / "gradlew.bat"
# wrapper_jar = (
#     source_dir
#     / "gradle"
#     / "wrapper"
#     / "gradle-wrapper.jar"
# )
# wrapper_properties = (
#     source_dir
#     / "gradle"
#     / "wrapper"
#     / "gradle-wrapper.properties"
# )

# print("\nEnvironment checks")
# print("=" * 60)
# print("Source directory exists:", source_dir.is_dir())
# print("gradlew.bat exists:      ", wrapper_path.is_file())
# print("Wrapper JAR exists:      ", wrapper_jar.is_file())
# print("Wrapper properties exist:", wrapper_properties.is_file())
# print("java path:               ", shutil.which("java"))
# print("javac path:              ", shutil.which("javac"))
# print("JAVA_HOME:               ", os.environ.get("JAVA_HOME"))
# print("COMSPEC:                 ", os.environ.get("COMSPEC"))

# if wrapper_properties.is_file():
#     print("\ngradle-wrapper.properties")
#     print("=" * 60)
#     print(
#         wrapper_properties.read_text(
#             encoding="utf-8",
#             errors="replace",
#         )
#     )


# print("\nDirect Gradle wrapper test")
# print("=" * 60)

# cmd_path = (
#     os.environ.get("COMSPEC")
#     or shutil.which("cmd.exe")
# )

# result = subprocess.run(
#     [
#         cmd_path,
#         "/d",
#         "/c",
#         "call",
#         str(wrapper_path),
#         "--no-daemon",
#         "--console=plain",
#         "--version",
#     ],
#     cwd=str(source_dir),
#     capture_output=True,
#     text=True,
#     encoding="utf-8",
#     errors="replace",
#     check=False,
# )

# print("Return code:", result.returncode)

# print("\nSTDOUT")
# print(result.stdout or "[empty]")

# print("\nSTDERR")
# print(result.stderr or "[empty]")

Saved Gradle log
Command: gradlew.bat --no-daemon --console=plain --version
Working directory: C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\tools\usgs_nshmp\nshmp-haz-2.6.5\nshmp-haz-2.6.5
Started UTC: 2026-07-30T00:13:05.775543+00:00
Finished UTC: 2026-07-30T00:13:05.870722+00:00
Return code: 1
'gradlew.bat' is not recognized as an internal or external command,
operable program or batch file.


Environment checks
Source directory exists: True
gradlew.bat exists:       True
Wrapper JAR exists:       True
Wrapper properties exist: True
java path:                C:\Program Files\Eclipse Adoptium\jdk-11.0.32.9-hotspot\bin\java.EXE
javac path:               C:\Program Files\Eclipse Adoptium\jdk-11.0.32.9-hotspot\bin\javac.EXE
JAVA_HOME:                C:\Program Files\Eclipse Adoptium\jdk-11.0.32.9-hotspot\
COMSPEC:                  C:\Windows\system32\cmd.exe

gradle-wrapper.properties
distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
distributionUrl=ht

In [8]:
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import os
import shutil
import subprocess
import zipfile

import pandas as pd
from IPython.display import display


source_dir = Path(NSHMP_HAZ_SOURCE_DIR).resolve()

if not source_dir.is_dir():
    raise FileNotFoundError(
        f"nshmp-haz source directory not found:\n{source_dir}"
    )

if "run_gradle_command" not in globals():
    raise RuntimeError(
        "The Gradle helper function is unavailable. "
        "Run Cell 3 before running Cell 4."
    )


api_dir = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_api"
)

api_dir.mkdir(
    parents=True,
    exist_ok=True,
)

runtime_log_path = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_runtime_classpath.log"
)

dependency_log_path = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_runtime_dependencies.log"
)

runtime_manifest_path = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_runtime_classpath.csv"
)

class_inventory_path = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_model_class_inventory.csv"
)

api_summary_path = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_api_summary.csv"
)

api_metadata_path = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_api_metadata.json"
)

init_script_path = (
    JAVA_EXPORTER_DIR
    / "print_runtime_classpath.gradle"
)


init_script_text = """
gradle.projectsEvaluated {
    def project = gradle.rootProject

    if (project.plugins.hasPlugin("java")) {
        project.tasks.register("printRuntimeClasspathForNotebook") {
            doLast {
                project.sourceSets.main.runtimeClasspath.files.each { file ->
                    println "RUNTIME_CP\\t${file.absolutePath}"
                }
            }
        }
    }
}
""".strip()


init_script_path.write_text(
    init_script_text + "\n",
    encoding="utf-8",
)


print("Resolving the Gradle runtime classpath...")

runtime_result = run_gradle_command(
    [
        "--no-daemon",
        "--console=plain",
        "--init-script",
        str(init_script_path),
        "printRuntimeClasspathForNotebook",
    ],
    runtime_log_path,
)


if runtime_result.returncode != 0:
    print(runtime_result.stdout)

    raise RuntimeError(
        "Gradle could not resolve the runtime classpath.\n\n"
        f"Complete log:\n{runtime_log_path}"
    )


runtime_paths = []

for line in runtime_result.stdout.splitlines():
    if not line.startswith("RUNTIME_CP\t"):
        continue

    path_text = line.split(
        "\t",
        maxsplit=1,
    )[1].strip()

    path = Path(path_text).resolve()

    if path.exists():
        runtime_paths.append(path)


runtime_paths = list(
    dict.fromkeys(runtime_paths)
)


if not runtime_paths:
    raise RuntimeError(
        "Gradle completed successfully, but no runtime classpath "
        "entries were identified."
    )


dependency_result = run_gradle_command(
    [
        "--no-daemon",
        "--console=plain",
        "dependencies",
        "--configuration",
        "runtimeClasspath",
    ],
    dependency_log_path,
)


if dependency_result.returncode != 0:
    print(dependency_result.stdout)

    raise RuntimeError(
        "Gradle could not produce the runtime dependency report.\n\n"
        f"Complete log:\n{dependency_log_path}"
    )


def sha256_file(file_path):
    digest = hashlib.sha256()

    with file_path.open("rb") as file_object:
        for block in iter(
            lambda: file_object.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


runtime_records = []

for path in runtime_paths:
    if path.is_dir():
        path_type = "directory"
        size_bytes = None
        checksum = None

    elif path.suffix.lower() == ".jar":
        path_type = "jar"
        size_bytes = path.stat().st_size
        checksum = sha256_file(path)

    else:
        path_type = "file"
        size_bytes = path.stat().st_size
        checksum = sha256_file(path)

    runtime_records.append(
        {
            "path": str(path),
            "name": path.name,
            "type": path_type,
            "size_bytes": size_bytes,
            "sha256": checksum,
        }
    )


runtime_classpath = pd.DataFrame(
    runtime_records
)

runtime_classpath.to_csv(
    runtime_manifest_path,
    index=False,
)


print("\nRuntime classpath:")
display(runtime_classpath)


dependency_lines = [
    line.strip()
    for line in dependency_result.stdout.splitlines()
    if "nshmp-lib" in line.lower()
]


print("\nnshmp-lib dependency entries:")

if dependency_lines:
    for line in dependency_lines:
        print(line)
else:
    print("No nshmp-lib entry was identified in the dependency report.")


project_jar_dir = source_dir / "build" / "libs"

project_jars = (
    sorted(project_jar_dir.glob("*.jar"))
    if project_jar_dir.is_dir()
    else []
)

dependency_jars = [
    path
    for path in runtime_paths
    if path.is_file()
    and path.suffix.lower() == ".jar"
]

all_jars = list(
    dict.fromkeys(
        [
            *project_jars,
            *dependency_jars,
        ]
    )
)


if not all_jars:
    raise RuntimeError(
        "No project or dependency JAR files were found."
    )


class_records = []

for jar_path in all_jars:
    if not zipfile.is_zipfile(jar_path):
        continue

    with zipfile.ZipFile(
        jar_path,
        mode="r",
    ) as jar_file:
        entries = jar_file.namelist()

    for entry in entries:
        if not entry.endswith(".class"):
            continue

        if "$" in entry:
            continue

        class_name = (
            entry[:-6]
            .replace("/", ".")
        )

        simple_name = class_name.rsplit(
            ".",
            maxsplit=1,
        )[-1]

        class_records.append(
            {
                "simple_name": simple_name,
                "class_name": class_name,
                "jar_name": jar_path.name,
                "jar_path": str(jar_path),
            }
        )


all_classes = pd.DataFrame(
    class_records
)

if all_classes.empty:
    raise RuntimeError(
        "No Java classes were found in the resolved JAR files."
    )


keywords = [
    "HazardModel",
    "SourceTree",
    "SourceSet",
    "RuptureSet",
    "Rupture",
    "GridSource",
    "InterfaceSource",
    "FaultSource",
    "SystemSource",
    "SourceType",
]


relevant_classes = all_classes.loc[
    all_classes["simple_name"].apply(
        lambda name: any(
            keyword.lower() in name.lower()
            for keyword in keywords
        )
    )
].copy()


relevant_classes = relevant_classes.sort_values(
    [
        "simple_name",
        "class_name",
        "jar_name",
    ]
).reset_index(
    drop=True
)


relevant_classes.to_csv(
    class_inventory_path,
    index=False,
)


print("\nRelevant USGS model classes:")
display(relevant_classes)


hazard_model_matches = all_classes.loc[
    all_classes["simple_name"]
    == "HazardModel"
].copy()


if hazard_model_matches.empty:
    raise RuntimeError(
        "HazardModel was not found in the project or dependency JARs."
    )


print("\nHazardModel location:")
display(hazard_model_matches)


nshmp_lib_jars = [
    path
    for path in all_jars
    if "nshmp-lib" in path.name.lower()
]


print("\nResolved nshmp-lib JAR files:")

for jar_path in nshmp_lib_jars:
    print(jar_path)


javap_path = shutil.which("javap")

if javap_path is None:
    java_path = shutil.which("java")

    if java_path:
        candidate = (
            Path(java_path).resolve().parent
            / "javap.exe"
        )

        if candidate.is_file():
            javap_path = str(candidate)


if javap_path is None:
    raise FileNotFoundError(
        "javap could not be located. Confirm that the JDK bin "
        "directory is available through PATH."
    )


preferred_simple_names = [
    "HazardModel",
    "SourceTree",
    "SourceSet",
    "RuptureSet",
    "Rupture",
    "Source",
    "SourceType",
    "GridSourceSet",
    "InterfaceSourceSet",
    "FaultSourceSet",
    "SystemSourceSet",
    "GridSource",
    "InterfaceSource",
]


inspection_targets = []

for simple_name in preferred_simple_names:
    matches = relevant_classes.loc[
        relevant_classes["simple_name"]
        == simple_name
    ]

    for row in matches.itertuples():
        inspection_targets.append(
            {
                "simple_name": row.simple_name,
                "class_name": row.class_name,
                "jar_path": Path(row.jar_path),
            }
        )


if not inspection_targets:
    raise RuntimeError(
        "No relevant USGS classes were selected for API inspection."
    )


inspection_records = []

print("\nSelected public API signatures:")

for target in inspection_targets:
    simple_name = target["simple_name"]
    class_name = target["class_name"]
    jar_path = target["jar_path"]

    result = subprocess.run(
        [
            javap_path,
            "-public",
            "-classpath",
            str(jar_path),
            class_name,
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )

    output = "\n".join(
        value.strip()
        for value in [
            result.stdout,
            result.stderr,
        ]
        if value and value.strip()
    )

    safe_name = class_name.replace(
        ".",
        "_",
    )

    output_path = (
        api_dir
        / f"{safe_name}.txt"
    )

    output_path.write_text(
        output + "\n",
        encoding="utf-8",
    )

    public_lines = [
        line.strip()
        for line in output.splitlines()
        if line.strip().startswith(
            (
                "public ",
                "protected ",
            )
        )
    ]

    inspection_records.append(
        {
            "simple_name": simple_name,
            "class_name": class_name,
            "jar_name": jar_path.name,
            "return_code": result.returncode,
            "inspection_passed": result.returncode == 0,
            "public_signature_count": len(public_lines),
            "output_file": str(
                output_path.relative_to(
                    PROJECT_ROOT
                )
            ),
        }
    )

    print(f"\n{class_name}")
    print("-" * len(class_name))

    if public_lines:
        for line in public_lines:
            print(line)
    else:
        print(output)


api_summary = pd.DataFrame(
    inspection_records
)

api_summary.to_csv(
    api_summary_path,
    index=False,
)


metadata = {
    "nshmp_haz_tag": NSHMP_HAZ_TAG,
    "nshmp_haz_commit": NSHMP_HAZ_COMMIT,
    "runtime_classpath_entries": len(runtime_paths),
    "runtime_jar_count": len(dependency_jars),
    "project_jar_count": len(project_jars),
    "nshmp_lib_jars": [
        str(path)
        for path in nshmp_lib_jars
    ],
    "hazard_model_classes": (
        hazard_model_matches[
            "class_name"
        ].tolist()
    ),
    "classes_inspected": (
        api_summary[
            "class_name"
        ].tolist()
    ),
    "runtime_manifest": str(
        runtime_manifest_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "class_inventory": str(
        class_inventory_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "api_summary": str(
        api_summary_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


api_metadata_path.write_text(
    json.dumps(
        metadata,
        indent=2,
    ),
    encoding="utf-8",
)


validation = pd.DataFrame(
    [
        {
            "check": "Runtime classpath resolved",
            "passes": len(runtime_paths) > 0,
        },
        {
            "check": "Runtime dependency JARs found",
            "passes": len(dependency_jars) > 0,
        },
        {
            "check": "nshmp-lib JAR found",
            "passes": len(nshmp_lib_jars) > 0,
        },
        {
            "check": "HazardModel found",
            "passes": not hazard_model_matches.empty,
        },
        {
            "check": "Relevant model classes found",
            "passes": len(relevant_classes) > 0,
        },
        {
            "check": "API inspections passed",
            "passes": api_summary[
                "inspection_passed"
            ].all(),
        },
        {
            "check": "Runtime manifest created",
            "passes": runtime_manifest_path.is_file(),
        },
        {
            "check": "Class inventory created",
            "passes": class_inventory_path.is_file(),
        },
        {
            "check": "API summary created",
            "passes": api_summary_path.is_file(),
        },
        {
            "check": "API metadata created",
            "passes": api_metadata_path.is_file(),
        },
    ]
)


print("\nCell 4 validation:")
display(validation)


if not validation["passes"].all():
    failed_checks = validation.loc[
        ~validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Cell 4 validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed_checks
        )
    )


print("\nCELL 4 VALIDATION PASSED")

print(f"\nRuntime entries:       {len(runtime_paths):,}")
print(f"Dependency JARs:       {len(dependency_jars):,}")
print(f"nshmp-lib JARs:        {len(nshmp_lib_jars):,}")
print(f"Relevant classes:      {len(relevant_classes):,}")
print(f"Classes inspected:     {len(api_summary):,}")
print(f"Runtime manifest:      {runtime_manifest_path}")
print(f"Class inventory:       {class_inventory_path}")
print(f"API summary:           {api_summary_path}")

print(
    "\nThe exact USGS runtime dependencies and model APIs "
    "have been identified."
)

print(
    "\nNext step: create the rupture-rate exporter using "
    "the verified class signatures."
)

Resolving the Gradle runtime classpath...

Runtime classpath:


,path,name,type,size_bytes,sha256
0,C:\Users\USER\Documents\GitHub\seismic-correla...,main,directory,NaN,None
1,C:\Users\USER\Documents\GitHub\seismic-correla...,main,directory,NaN,None
2,C:\Users\USER\.gradle\caches\modules-2\files-2...,nshmp-lib-1.7.3.jar,jar,4021573.0,4cbc3ca7268134cf2d0ee5f62b6a6726666006e95d78ce...
3,C:\Users\USER\.gradle\caches\modules-2\files-2...,org.eclipse.jgit-6.7.0.202309050840-r.jar,jar,3139434.0,b564477d092241aaab50c84ac5dd1ac375c0182044d7e5...
4,C:\Users\USER\.gradle\caches\modules-2\files-2...,micronaut-openapi-4.8.7.jar,jar,2792260.0,a8a33c06bd71fee3a30f2bba0dc14c974b4f587eb6d25b...
...,...,...,...,...,...
156,C:\Users\USER\.gradle\caches\modules-2\files-2...,flexmark-util-sequence-0.62.2.jar,jar,222870.0,6684a0048ad088452419a2871a6516e7fd3013700cb34a...
157,C:\Users\USER\.gradle\caches\modules-2\files-2...,flexmark-util-collection-0.62.2.jar,jar,66693.0,59f350f064aeb3d0e01e97fb773fb9701e3605d9db7a6e...
158,C:\Users\USER\.gradle\caches\modules-2\files-2...,flexmark-util-data-0.62.2.jar,jar,24936.0,4ec42683f8ae51ee8227f3443a54ef0d70d076b593890a...
159,C:\Users\USER\.gradle\caches\modules-2\files-2...,flexmark-util-misc-0.62.2.jar,jar,59210.0,06cec0698633f875e668b401dbab208e0f56a5d55f956a...



nshmp-lib dependency entries:
+--- ghsc:nshmp-lib:1.7.3

Relevant USGS model classes:


,simple_name,class_name,jar_name,jar_path
0,AbstractRuptureSet,gov.usgs.earthquake.nshmp.model.AbstractRuptur...,nshmp-lib-1.7.3.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...
1,ClusterRuptureSet,gov.usgs.earthquake.nshmp.model.ClusterRuptureSet,nshmp-lib-1.7.3.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...
2,ComplianceResourceTypeListCopier,software.amazon.awssdk.services.ssm.model.Comp...,ssm-2.29.52.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...
3,CredentialSourceType,software.amazon.awssdk.auth.credentials.intern...,auth-2.29.52.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...
4,FaultRuptureSet,gov.usgs.earthquake.nshmp.model.FaultRuptureSet,nshmp-lib-1.7.3.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...
5,FaultSource,gov.usgs.earthquake.nshmp.model.FaultSource,nshmp-lib-1.7.3.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...
6,GridRuptureSet,gov.usgs.earthquake.nshmp.model.GridRuptureSet,nshmp-lib-1.7.3.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...
7,GridSourcePlanar,gov.usgs.earthquake.nshmp.model.GridSourcePlanar,nshmp-lib-1.7.3.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...
8,HazardModel,gov.usgs.earthquake.nshmp.model.HazardModel,nshmp-lib-1.7.3.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...
9,InterfaceRuptureSet,gov.usgs.earthquake.nshmp.model.InterfaceRuptu...,nshmp-lib-1.7.3.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...



HazardModel location:


,simple_name,class_name,jar_name,jar_path
202,HazardModel,gov.usgs.earthquake.nshmp.model.HazardModel,nshmp-lib-1.7.3.jar,C:\Users\USER\.gradle\caches\modules-2\files-2...



Resolved nshmp-lib JAR files:
C:\Users\USER\.gradle\caches\modules-2\files-2.1\ghsc\nshmp-lib\1.7.3\8bfb8fabae65995ba989c8b87b1741e11035e608\nshmp-lib-1.7.3.jar

Selected public API signatures:

gov.usgs.earthquake.nshmp.model.HazardModel
-------------------------------------------
public final class gov.usgs.earthquake.nshmp.model.HazardModel implements java.lang.Iterable<gov.usgs.earthquake.nshmp.model.SourceTree> {
public static gov.usgs.earthquake.nshmp.model.HazardModel load(java.nio.file.Path);
public int size();
public java.util.Iterator<gov.usgs.earthquake.nshmp.model.SourceTree> iterator();
public java.nio.file.Path root();
public java.lang.String name();
public gov.usgs.earthquake.nshmp.calc.CalcConfig config();
public java.util.Set<gov.usgs.earthquake.nshmp.gmm.Gmm> gmms();
public java.util.Set<gov.usgs.earthquake.nshmp.model.SourceType> types();
public java.util.Set<gov.usgs.earthquake.nshmp.model.TectonicSetting> settings();
public java.util.Optional<gov.usgs.earthquake.n

,check,passes
0,Runtime classpath resolved,True
1,Runtime dependency JARs found,True
2,nshmp-lib JAR found,True
3,HazardModel found,True
4,Relevant model classes found,True
5,API inspections passed,True
6,Runtime manifest created,True
7,Class inventory created,True
8,API summary created,True
9,API metadata created,True



CELL 4 VALIDATION PASSED

Runtime entries:       161
Dependency JARs:       159
nshmp-lib JARs:        1
Relevant classes:      29
Classes inspected:     7
Runtime manifest:      C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_2_6_5_runtime_classpath.csv
Class inventory:       C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_2_6_5_model_class_inventory.csv
API summary:           C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_2_6_5_api_summary.csv

The exact USGS runtime dependencies and model APIs have been identified.

Next step: create the rupture-rate exporter using the verified class signatures.


In [13]:
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import os
import shutil
import subprocess

import pandas as pd
from IPython.display import display


exporter_dir = Path(JAVA_EXPORTER_DIR).resolve()
source_dir = exporter_dir / "src"
classes_dir = exporter_dir / "classes"

source_dir.mkdir(
    parents=True,
    exist_ok=True,
)

if classes_dir.exists():
    shutil.rmtree(classes_dir)

classes_dir.mkdir(
    parents=True,
    exist_ok=True,
)


runtime_manifest = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_runtime_classpath.csv"
)

if not runtime_manifest.is_file():
    raise FileNotFoundError(
        "The runtime classpath manifest was not found. "
        "Run Cell 4 before Cell 5."
    )


runtime_table = pd.read_csv(
    runtime_manifest
)

runtime_paths = [
    Path(path).resolve()
    for path in runtime_table["path"]
    if Path(path).exists()
]

if not runtime_paths:
    raise RuntimeError(
        "No valid runtime classpath entries were found."
    )


classpath = os.pathsep.join(
    str(path)
    for path in runtime_paths
)


java_source_path = (
    source_dir
    / "RuptureRateExporter.java"
)

class_file_path = (
    classes_dir
    / "RuptureRateExporter.class"
)

compile_log_path = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_exporter_compile.log"
)

compile_metadata_path = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_exporter_compile_metadata.json"
)

javac_argument_file = (
    exporter_dir
    / "javac_arguments.txt"
)


java_source = r'''
import gov.usgs.earthquake.nshmp.fault.surface.RuptureSurface;
import gov.usgs.earthquake.nshmp.geo.Location;
import gov.usgs.earthquake.nshmp.model.HazardModel;
import gov.usgs.earthquake.nshmp.model.Rupture;
import gov.usgs.earthquake.nshmp.model.RuptureSet;
import gov.usgs.earthquake.nshmp.model.Source;
import gov.usgs.earthquake.nshmp.model.SourceTree;
import gov.usgs.earthquake.nshmp.tree.Branch;

import java.io.PrintWriter;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;
import java.security.MessageDigest;
import java.time.Instant;
import java.util.Locale;

public final class RuptureRateExporter {

  private static final String HEADER = String.join(",",
      "rupture_id",
      "target_group",
      "model_name",
      "tree_id",
      "tree_name",
      "tree_path",
      "tree_type",
      "tectonic_setting",
      "branch_index",
      "branch_weight",
      "rupture_set_id",
      "rupture_set_name",
      "rupture_set_type",
      "rupture_set_weight",
      "source_index",
      "source_id",
      "source_name",
      "source_type",
      "rupture_index",
      "magnitude",
      "rake_deg",
      "raw_annual_rate",
      "weighted_annual_rate",
      "centroid_longitude",
      "centroid_latitude",
      "centroid_depth_km",
      "top_depth_km",
      "strike_deg",
      "dip_deg",
      "dip_direction_deg",
      "length_km",
      "width_km",
      "area_km2"
  );

  private RuptureRateExporter() {}

  public static void main(String[] args) throws Exception {

    if (args.length != 2) {
      System.err.println(
          "Usage: RuptureRateExporter <model-directory> <output-csv>");
      System.exit(2);
    }

    Path modelPath = Path.of(args[0]).toAbsolutePath().normalize();
    Path outputPath = Path.of(args[1]).toAbsolutePath().normalize();

    if (!Files.isDirectory(modelPath)) {
      throw new IllegalArgumentException(
          "Model directory does not exist: " + modelPath);
    }

    if (outputPath.getParent() != null) {
      Files.createDirectories(outputPath.getParent());
    }

    System.out.println("Loading model: " + modelPath);
    System.out.println("Started: " + Instant.now());

    HazardModel model = HazardModel.load(modelPath);

    long selectedTreeCount = 0;
    long ruptureSetCount = 0;
    long sourceCount = 0;
    long ruptureCount = 0;
    long interfaceCount = 0;
    long slabCount = 0;

    double rawRateSum = 0.0;
    double weightedRateSum = 0.0;

    try (
        PrintWriter writer = new PrintWriter(
            Files.newBufferedWriter(
                outputPath,
                StandardCharsets.UTF_8))
    ) {

      writer.println(HEADER);

      for (SourceTree tree : model) {

        String treePath = normalizePath(
            tree.path().toString());

        String treeType = tree.type().name();

        String targetGroup = targetGroup(
            treeType,
            treePath);

        if (targetGroup == null) {
          continue;
        }

        selectedTreeCount++;

        System.out.println(
            "Selected tree: "
            + treeType
            + " | "
            + treePath);

        for (
            int branchIndex = 0;
            branchIndex < tree.size();
            branchIndex++
        ) {

          Branch<RuptureSet<? extends Source>> branch =
              tree.get(branchIndex);

          RuptureSet<? extends Source> ruptureSet =
              branch.value();

          double branchWeight = branch.weight();
          double ruptureSetWeight = ruptureSet.weight();

          ruptureSetCount++;

          for (
              int sourceIndex = 0;
              sourceIndex < ruptureSet.size();
              sourceIndex++
          ) {

            Source source = ruptureSet.get(
                sourceIndex);

            sourceCount++;

            for (
                int ruptureIndex = 0;
                ruptureIndex < source.size();
                ruptureIndex++
            ) {

              Rupture rupture = source.get(
                  ruptureIndex);

              double rawRate = rupture.rate();

              // The rupture-set weight is applied once.
              double weightedRate =
                  rawRate * ruptureSetWeight;

              SurfaceValues geometry =
                  SurfaceValues.from(
                      rupture.surface());

              String stableKey = String.join("|",
                  model.name(),
                  Integer.toString(tree.id()),
                  treePath,
                  Integer.toString(branchIndex),
                  Integer.toString(ruptureSet.id()),
                  Integer.toString(sourceIndex),
                  Integer.toString(source.id()),
                  Integer.toString(ruptureIndex),
                  Double.toString(rupture.mag()),
                  Double.toString(rupture.rake()),
                  number(geometry.longitude),
                  number(geometry.latitude),
                  number(geometry.centroidDepth)
              );

              String ruptureId = sha256(
                  stableKey).substring(0, 24);

              writer.println(String.join(",",
                  csv(ruptureId),
                  csv(targetGroup),
                  csv(model.name()),
                  integer(tree.id()),
                  csv(tree.name()),
                  csv(treePath),
                  csv(treeType),
                  csv(tree.setting().name()),
                  integer(branchIndex),
                  number(branchWeight),
                  integer(ruptureSet.id()),
                  csv(ruptureSet.name()),
                  csv(ruptureSet.type().name()),
                  number(ruptureSetWeight),
                  integer(sourceIndex),
                  integer(source.id()),
                  csv(source.name()),
                  csv(source.type().name()),
                  integer(ruptureIndex),
                  number(rupture.mag()),
                  number(rupture.rake()),
                  number(rawRate),
                  number(weightedRate),
                  number(geometry.longitude),
                  number(geometry.latitude),
                  number(geometry.centroidDepth),
                  number(geometry.topDepth),
                  number(geometry.strike),
                  number(geometry.dip),
                  number(geometry.dipDirection),
                  number(geometry.length),
                  number(geometry.width),
                  number(geometry.area)
              ));

              ruptureCount++;
              rawRateSum += rawRate;
              weightedRateSum += weightedRate;

              if (
                  targetGroup.equals(
                      "cascadia_interface")
              ) {
                interfaceCount++;
              } else {
                slabCount++;
              }
            }
          }
        }
      }
    }

    if (selectedTreeCount == 0) {
      throw new IllegalStateException(
          "No Cascadia interface or Oregon slab trees were selected.");
    }

    if (ruptureCount == 0) {
      throw new IllegalStateException(
          "The selected source trees produced no ruptures.");
    }

    System.out.println();
    System.out.println("EXPORT_COMPLETE");
    System.out.println(
        "selected_trees=" + selectedTreeCount);
    System.out.println(
        "rupture_sets=" + ruptureSetCount);
    System.out.println(
        "sources=" + sourceCount);
    System.out.println(
        "ruptures=" + ruptureCount);
    System.out.println(
        "cascadia_interface_ruptures="
        + interfaceCount);
    System.out.println(
        "oregon_intraslab_ruptures="
        + slabCount);
    System.out.println(
        "raw_rate_sum="
        + String.format(
            Locale.US,
            "%.17g",
            rawRateSum));
    System.out.println(
        "weighted_rate_sum="
        + String.format(
            Locale.US,
            "%.17g",
            weightedRateSum));
    System.out.println(
        "output=" + outputPath);
    System.out.println(
        "finished=" + Instant.now());
  }

  private static String targetGroup(
    String treeType,
    String treePath
) {

  String path = treePath.toLowerCase(
      Locale.US);

  if (
      treeType.equals("INTERFACE")
      && path.contains("/subduction/interface/cascadia")
  ) {
    return "cascadia_interface";
  }

  if (
      treeType.equals("SLAB")
      && path.contains("/subduction/slab/or")
  ) {
    return "oregon_intraslab";
  }

  return null;
}

  private static String normalizePath(
      String value
  ) {
    return value.replace('\\', '/');
  }

  private static String integer(
      int value
  ) {
    return Integer.toString(value);
  }

  private static String number(
      double value
  ) {

    if (!Double.isFinite(value)) {
      return "";
    }

    return String.format(
        Locale.US,
        "%.17g",
        value);
  }

  private static String csv(
      String value
  ) {

    if (value == null) {
      return "";
    }

    boolean requiresQuotes =
        value.contains(",")
        || value.contains("\"")
        || value.contains("\n")
        || value.contains("\r");

    if (!requiresQuotes) {
      return value;
    }

    return "\""
        + value.replace(
            "\"",
            "\"\"")
        + "\"";
  }

  private static String sha256(
      String value
  ) throws Exception {

    MessageDigest digest =
        MessageDigest.getInstance(
            "SHA-256");

    byte[] bytes = digest.digest(
        value.getBytes(
            StandardCharsets.UTF_8));

    StringBuilder output =
        new StringBuilder();

    for (byte current : bytes) {
      output.append(
          String.format(
              Locale.US,
              "%02x",
              current & 0xff));
    }

    return output.toString();
  }

  private static final class SurfaceValues {

    double longitude = Double.NaN;
    double latitude = Double.NaN;
    double centroidDepth = Double.NaN;
    double topDepth = Double.NaN;
    double strike = Double.NaN;
    double dip = Double.NaN;
    double dipDirection = Double.NaN;
    double length = Double.NaN;
    double width = Double.NaN;
    double area = Double.NaN;

    static SurfaceValues from(
        RuptureSurface surface
    ) {

      SurfaceValues values =
          new SurfaceValues();

      if (surface == null) {
        return values;
      }

      try {
        Location centroid =
            surface.centroid();

        if (centroid != null) {
          values.longitude =
              centroid.longitude;

          values.latitude =
              centroid.latitude;

          values.centroidDepth =
              centroid.depth;
        }
      } catch (RuntimeException ignored) {
      }

      try {
        values.topDepth =
            surface.depth();
      } catch (RuntimeException ignored) {
      }

      try {
        values.strike =
            surface.strike();
      } catch (RuntimeException ignored) {
      }

      try {
        values.dip =
            surface.dip();
      } catch (RuntimeException ignored) {
      }

      try {
        values.dipDirection =
            surface.dipDirection();
      } catch (RuntimeException ignored) {
      }

      try {
        values.length =
            surface.length();
      } catch (RuntimeException ignored) {
      }

      try {
        values.width =
            surface.width();
      } catch (RuntimeException ignored) {
      }

      try {
        values.area =
            surface.area();
      } catch (RuntimeException ignored) {
      }

      return values;
    }
  }
}
'''.strip()


java_source_path.write_text(
    java_source + "\n",
    encoding="utf-8",
)


javac_path = shutil.which("javac")

if javac_path is None:
    raise FileNotFoundError(
        "javac was not found through PATH."
    )


def argument_file_value(value):
    escaped = str(value).replace(
        "\\",
        "\\\\",
    )

    escaped = escaped.replace(
        '"',
        '\\"',
    )

    return f'"{escaped}"'


javac_arguments = "\n".join(
    [
        "-encoding",
        "UTF-8",
        "-classpath",
        argument_file_value(classpath),
        "-d",
        argument_file_value(classes_dir),
        argument_file_value(java_source_path),
    ]
)

javac_argument_file.write_text(
    javac_arguments + "\n",
    encoding="utf-8",
)


compile_started = datetime.now(
    timezone.utc
)

compile_result = subprocess.run(
    [
        javac_path,
        f"@{javac_argument_file}",
    ],
    cwd=str(exporter_dir),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)

compile_finished = datetime.now(
    timezone.utc
)


compile_output = "\n".join(
    output.strip()
    for output in [
        compile_result.stdout,
        compile_result.stderr,
    ]
    if output and output.strip()
)


compile_log_path.write_text(
    "\n".join(
        [
            f"Started UTC: {compile_started.isoformat()}",
            f"Finished UTC: {compile_finished.isoformat()}",
            f"Return code: {compile_result.returncode}",
            "",
            compile_output,
            "",
        ]
    ),
    encoding="utf-8",
)


print("Java compiler output:")

if compile_output:
    print(compile_output)
else:
    print("[no compiler messages]")


def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as file_object:
        for block in iter(
            lambda: file_object.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


validation = pd.DataFrame(
    [
        {
            "check": "Java source created",
            "passes": java_source_path.is_file(),
        },
        {
            "check": "Runtime classpath available",
            "passes": len(runtime_paths) > 0,
        },
        {
            "check": "javac completed successfully",
            "passes": compile_result.returncode == 0,
        },
        {
            "check": "Exporter class created",
            "passes": class_file_path.is_file(),
        },
        {
            "check": "Compile log created",
            "passes": compile_log_path.is_file(),
        },
    ]
)


print("\nCell 5 validation:")
display(validation)


if not validation["passes"].all():
    failed_checks = validation.loc[
        ~validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Exporter compilation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed_checks
        )
        + f"\n\nCompile log:\n{compile_log_path}"
    )


compile_metadata = {
    "exporter_class": "RuptureRateExporter",
    "java_source": str(
        java_source_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "compiled_class": str(
        class_file_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "source_sha256": file_sha256(
        java_source_path
    ),
    "class_sha256": file_sha256(
        class_file_path
    ),
    "runtime_classpath_entries": len(
        runtime_paths
    ),
    "javac_path": javac_path,
    "compile_return_code": (
        compile_result.returncode
    ),
    "compile_started_at_utc": (
        compile_started.isoformat()
    ),
    "compile_finished_at_utc": (
        compile_finished.isoformat()
    ),
    "compile_log": str(
        compile_log_path.relative_to(
            PROJECT_ROOT
        )
    ),
}


compile_metadata_path.write_text(
    json.dumps(
        compile_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


print("\nCELL 5 VALIDATION PASSED")

print(f"\nJava source:          {java_source_path}")
print(f"Compiled class:       {class_file_path}")
print(f"Classpath entries:    {len(runtime_paths):,}")
print(
    "Source SHA-256:      "
    f"{compile_metadata['source_sha256']}"
)
print(
    "Class SHA-256:       "
    f"{compile_metadata['class_sha256']}"
)
print(f"Compile log:          {compile_log_path}")

print(
    "\nThe rupture-rate exporter compiled successfully "
    "against the exact USGS runtime libraries."
)

print(
    "\nNext step: run the exporter and validate the "
    "Cascadia and Oregon rupture-rate tables."
)

Java compiler output:
[no compiler messages]

Cell 5 validation:


,check,passes
0,Java source created,True
1,Runtime classpath available,True
2,javac completed successfully,True
3,Exporter class created,True
4,Compile log created,True



CELL 5 VALIDATION PASSED

Java source:          C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\tools\usgs_rupture_rate_exporter\src\RuptureRateExporter.java
Compiled class:       C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\tools\usgs_rupture_rate_exporter\classes\RuptureRateExporter.class
Classpath entries:    161
Source SHA-256:      fbbb047e3108915b13144ddb98c0db287604ce5710a51eee0f09982afa417cec
Class SHA-256:       a40b579dabde29d56eb8693733a7a43bfcccd587bf5aebe63e49defaffffcbf7
Compile log:          C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_haz_2_6_5_exporter_compile.log

The rupture-rate exporter compiled successfully against the exact USGS runtime libraries.

Next step: run the exporter and validate the Cascadia and Oregon rupture-rate tables.


In [14]:
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import os
import shutil
import subprocess
import time

import numpy as np
import pandas as pd
from IPython.display import display


exporter_dir = Path(JAVA_EXPORTER_DIR).resolve()
exporter_classes_dir = exporter_dir / "classes"

exporter_class_file = (
    exporter_classes_dir
    / "RuptureRateExporter.class"
)

if not exporter_class_file.is_file():
    raise FileNotFoundError(
        "The compiled rupture exporter was not found. "
        "Run Cell 5 before Cell 6."
    )


runtime_manifest_path = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_runtime_classpath.csv"
)

if not runtime_manifest_path.is_file():
    raise FileNotFoundError(
        "The runtime classpath manifest was not found. "
        "Run Cell 4 before Cell 6."
    )


runtime_table = pd.read_csv(
    runtime_manifest_path
)

runtime_paths = []

for path_text in runtime_table["path"].dropna():
    path = Path(path_text).resolve()

    if path.exists():
        runtime_paths.append(path)


runtime_paths = list(
    dict.fromkeys(runtime_paths)
)

if not runtime_paths:
    raise RuntimeError(
        "No valid USGS runtime classpath entries were found."
    )


classpath_paths = [
    exporter_classes_dir,
    *runtime_paths,
]

classpath_paths = list(
    dict.fromkeys(classpath_paths)
)

classpath = os.pathsep.join(
    path.as_posix()
    for path in classpath_paths
)


raw_output_path = (
    RAW_EXPORT_DIR
    / "usgs_subduction_rupture_rates_raw.csv"
)

temporary_output_path = (
    RAW_EXPORT_DIR
    / "usgs_subduction_rupture_rates_raw.part.csv"
)

cascadia_output_path = (
    RUPTURE_RATE_DIR
    / "cascadia_interface_rupture_rates.csv"
)

oregon_output_path = (
    RUPTURE_RATE_DIR
    / "oregon_intraslab_rupture_rates.csv"
)

run_log_path = (
    METADATA_DIR
    / "usgs_rupture_rate_export.log"
)

summary_output_path = (
    METADATA_DIR
    / "usgs_rupture_rate_summary.csv"
)

validation_output_path = (
    METADATA_DIR
    / "usgs_rupture_rate_validation.csv"
)

run_metadata_path = (
    METADATA_DIR
    / "usgs_rupture_rate_export_metadata.json"
)

java_argument_path = (
    exporter_dir
    / "java_export_arguments.txt"
)


for output_path in [
    temporary_output_path,
]:
    if output_path.exists():
        output_path.unlink()


java_path = shutil.which("java")

if java_path is None:
    raise FileNotFoundError(
        "The java executable was not found through PATH."
    )


def quote_java_argument(value):
    text = str(value).replace(
        "\\",
        "/",
    )

    text = text.replace(
        '"',
        '\\"',
    )

    return f'"{text}"'


# An argument file avoids the Windows command-length limit.
java_arguments = "\n".join(
    [
        "-Xms512m",
        "-Xmx4g",
        "-Dfile.encoding=UTF-8",
        "-Djava.awt.headless=true",
        "-classpath",
        quote_java_argument(classpath),
        "RuptureRateExporter",
        quote_java_argument(MODEL_DIR),
        quote_java_argument(temporary_output_path),
    ]
)

java_argument_path.write_text(
    java_arguments + "\n",
    encoding="utf-8",
)


print("Running the rupture-rate exporter...")
print(f"Model:  {MODEL_DIR}")
print(f"Output: {raw_output_path}")
print(
    "\nThe model-loading step may be quiet for several minutes."
)


run_started = datetime.now(
    timezone.utc
)

start_time = time.perf_counter()

try:
    run_result = subprocess.run(
        [
            java_path,
            f"@{java_argument_path}",
        ],
        cwd=str(exporter_dir),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=7200,
        check=False,
    )

except subprocess.TimeoutExpired as error:
    partial_output = "\n".join(
        str(value)
        for value in [
            error.stdout,
            error.stderr,
        ]
        if value
    )

    run_log_path.write_text(
        partial_output,
        encoding="utf-8",
    )

    raise RuntimeError(
        "The rupture-rate export exceeded the two-hour timeout.\n\n"
        f"Partial log:\n{run_log_path}"
    ) from error


run_finished = datetime.now(
    timezone.utc
)

run_duration_seconds = (
    time.perf_counter()
    - start_time
)


run_output = "\n".join(
    output.strip()
    for output in [
        run_result.stdout,
        run_result.stderr,
    ]
    if output and output.strip()
)


run_log_path.write_text(
    "\n".join(
        [
            f"Started UTC: {run_started.isoformat()}",
            f"Finished UTC: {run_finished.isoformat()}",
            f"Duration seconds: {run_duration_seconds:.3f}",
            f"Return code: {run_result.returncode}",
            "",
            run_output,
            "",
        ]
    ),
    encoding="utf-8",
)


print("\nExporter output:")

output_lines = run_output.splitlines()

if len(output_lines) > 80:
    print(
        f"... {len(output_lines) - 80:,} earlier lines "
        "were saved to the log file ..."
    )

for line in output_lines[-80:]:
    print(line)


if run_result.returncode != 0:
    raise RuntimeError(
        "The rupture-rate exporter returned a nonzero status.\n\n"
        f"Return code: {run_result.returncode}\n"
        f"Complete log:\n{run_log_path}"
    )


if "EXPORT_COMPLETE" not in run_output:
    raise RuntimeError(
        "The Java process returned successfully, but the expected "
        "EXPORT_COMPLETE marker was not found.\n\n"
        f"Complete log:\n{run_log_path}"
    )


if not temporary_output_path.is_file():
    raise FileNotFoundError(
        "The exporter completed but did not create the expected CSV:\n"
        f"{temporary_output_path}"
    )


if temporary_output_path.stat().st_size == 0:
    raise RuntimeError(
        "The exporter created an empty output file."
    )


os.replace(
    temporary_output_path,
    raw_output_path,
)


print("\nReading the exported rupture table...")

ruptures = pd.read_csv(
    raw_output_path,
    low_memory=False,
)


required_columns = {
    "rupture_id",
    "target_group",
    "model_name",
    "tree_id",
    "tree_name",
    "tree_path",
    "tree_type",
    "tectonic_setting",
    "branch_index",
    "branch_weight",
    "rupture_set_id",
    "rupture_set_name",
    "rupture_set_type",
    "rupture_set_weight",
    "source_index",
    "source_id",
    "source_name",
    "source_type",
    "rupture_index",
    "magnitude",
    "rake_deg",
    "raw_annual_rate",
    "weighted_annual_rate",
}


missing_columns = sorted(
    required_columns
    - set(ruptures.columns)
)

if missing_columns:
    raise RuntimeError(
        "The exported table is missing required columns:\n"
        + "\n".join(
            f"  {column}"
            for column in missing_columns
        )
    )


numeric_columns = [
    "branch_weight",
    "rupture_set_weight",
    "magnitude",
    "rake_deg",
    "raw_annual_rate",
    "weighted_annual_rate",
]

for column in numeric_columns:
    ruptures[column] = pd.to_numeric(
        ruptures[column],
        errors="coerce",
    )


sort_columns = [
    "target_group",
    "tree_id",
    "tree_path",
    "branch_index",
    "rupture_set_id",
    "source_index",
    "rupture_index",
    "magnitude",
    "rupture_id",
]

ruptures = ruptures.sort_values(
    sort_columns,
    kind="stable",
).reset_index(
    drop=True
)


cascadia = ruptures.loc[
    ruptures["target_group"]
    == "cascadia_interface"
].copy()

oregon = ruptures.loc[
    ruptures["target_group"]
    == "oregon_intraslab"
].copy()


cascadia.to_csv(
    cascadia_output_path,
    index=False,
)

oregon.to_csv(
    oregon_output_path,
    index=False,
)


set_key = [
    "tree_id",
    "tree_path",
    "branch_index",
    "rupture_set_id",
]

source_key = [
    *set_key,
    "source_index",
    "source_id",
]


def summarize_group(group_name, data):
    return {
        "target_group": group_name,
        "rupture_rows": len(data),
        "source_trees": data[
            [
                "tree_id",
                "tree_path",
            ]
        ].drop_duplicates().shape[0],
        "rupture_sets": data[
            set_key
        ].drop_duplicates().shape[0],
        "sources": data[
            source_key
        ].drop_duplicates().shape[0],
        "minimum_magnitude": data[
            "magnitude"
        ].min(),
        "maximum_magnitude": data[
            "magnitude"
        ].max(),
        "raw_rate_sum": data[
            "raw_annual_rate"
        ].sum(),
        "weighted_rate_sum": data[
            "weighted_annual_rate"
        ].sum(),
        "unique_rakes": data[
            "rake_deg"
        ].nunique(),
        "source_types": ", ".join(
            sorted(
                data["source_type"]
                .dropna()
                .astype(str)
                .unique()
            )
        ),
    }


summary = pd.DataFrame(
    [
        summarize_group(
            "cascadia_interface",
            cascadia,
        ),
        summarize_group(
            "oregon_intraslab",
            oregon,
        ),
    ]
)

summary.to_csv(
    summary_output_path,
    index=False,
)


expected_weighted_rates = (
    ruptures["raw_annual_rate"]
    * ruptures["rupture_set_weight"]
)

rate_formula_matches = np.allclose(
    ruptures["weighted_annual_rate"],
    expected_weighted_rates,
    rtol=1e-12,
    atol=1e-18,
    equal_nan=False,
)

branch_and_set_weights_match = np.allclose(
    ruptures["branch_weight"],
    ruptures["rupture_set_weight"],
    rtol=1e-12,
    atol=1e-15,
    equal_nan=False,
)


oregon_source_counts = (
    oregon[
        [
            *set_key,
            "source_index",
            "source_id",
        ]
    ]
    .drop_duplicates()
    .groupby(
        set_key,
        dropna=False,
    )
    .size()
)


export_counts = {}

for line in run_output.splitlines():
    if "=" not in line:
        continue

    key, value = line.split(
        "=",
        maxsplit=1,
    )

    key = key.strip()
    value = value.strip()

    if key in {
        "selected_trees",
        "rupture_sets",
        "sources",
        "ruptures",
        "cascadia_interface_ruptures",
        "oregon_intraslab_ruptures",
    }:
        try:
            export_counts[key] = int(value)
        except ValueError:
            pass


expected_groups = {
    "cascadia_interface",
    "oregon_intraslab",
}

actual_groups = set(
    ruptures[
        "target_group"
    ].dropna()
)


core_numeric_values_are_finite = np.isfinite(
    ruptures[
        [
            "branch_weight",
            "rupture_set_weight",
            "magnitude",
            "raw_annual_rate",
            "weighted_annual_rate",
        ]
    ].to_numpy(
        dtype=float
    )
).all()


validation_records = [
    {
        "check": "Exporter completed successfully",
        "expected": True,
        "actual": run_result.returncode == 0,
        "passes": run_result.returncode == 0,
    },
    {
        "check": "Both target groups were exported",
        "expected": sorted(expected_groups),
        "actual": sorted(actual_groups),
        "passes": actual_groups == expected_groups,
    },
    {
        "check": "Exported rupture table is not empty",
        "expected": "> 0",
        "actual": len(ruptures),
        "passes": len(ruptures) > 0,
    },
    {
        "check": "Rupture identifiers are complete",
        "expected": 0,
        "actual": int(
            ruptures["rupture_id"].isna().sum()
        ),
        "passes": (
            ruptures["rupture_id"].isna().sum()
            == 0
        ),
    },
    {
        "check": "Rupture identifiers are unique",
        "expected": 0,
        "actual": int(
            ruptures["rupture_id"].duplicated().sum()
        ),
        "passes": (
            ruptures["rupture_id"].duplicated().sum()
            == 0
        ),
    },
    {
        "check": "Core numeric values are finite",
        "expected": True,
        "actual": core_numeric_values_are_finite,
        "passes": core_numeric_values_are_finite,
    },
    {
        "check": "Raw annual rates are positive",
        "expected": True,
        "actual": bool(
            (
                ruptures["raw_annual_rate"]
                > 0
            ).all()
        ),
        "passes": bool(
            (
                ruptures["raw_annual_rate"]
                > 0
            ).all()
        ),
    },
    {
        "check": "Weighted annual rates are positive",
        "expected": True,
        "actual": bool(
            (
                ruptures["weighted_annual_rate"]
                > 0
            ).all()
        ),
        "passes": bool(
            (
                ruptures["weighted_annual_rate"]
                > 0
            ).all()
        ),
    },
    {
        "check": "Rupture-set weights are in (0, 1]",
        "expected": True,
        "actual": bool(
            (
                (
                    ruptures[
                        "rupture_set_weight"
                    ]
                    > 0
                )
                & (
                    ruptures[
                        "rupture_set_weight"
                    ]
                    <= 1
                )
            ).all()
        ),
        "passes": bool(
            (
                (
                    ruptures[
                        "rupture_set_weight"
                    ]
                    > 0
                )
                & (
                    ruptures[
                        "rupture_set_weight"
                    ]
                    <= 1
                )
            ).all()
        ),
    },
    {
        "check": (
            "Weighted rate equals raw rate "
            "times rupture-set weight"
        ),
        "expected": True,
        "actual": rate_formula_matches,
        "passes": rate_formula_matches,
    },
    {
        "check": (
            "Branch and rupture-set weights agree"
        ),
        "expected": True,
        "actual": branch_and_set_weights_match,
        "passes": branch_and_set_weights_match,
    },
    {
        "check": "Cascadia rupture sets",
        "expected": 21,
        "actual": int(
            cascadia[
                set_key
            ].drop_duplicates().shape[0]
        ),
        "passes": (
            cascadia[
                set_key
            ].drop_duplicates().shape[0]
            == 21
        ),
    },
    {
    "check": "Oregon loaded rupture sets",
    "expected": 2,
    "actual": int(
        oregon[
            set_key
        ].drop_duplicates().shape[0]
    ),
    "passes": (
        oregon[
            set_key
        ].drop_duplicates().shape[0]
        == 2
    ),
},
    {
        "check": (
            "Each Oregon model combination "
            "contains 821 grid sources"
        ),
        "expected": 821,
        "actual": (
            sorted(
                oregon_source_counts
                .astype(int)
                .unique()
                .tolist()
            )
        ),
        "passes": bool(
            len(oregon_source_counts) == 2
            and (
                oregon_source_counts
                == 821
            ).all()
),
    },
    {
        "check": "Java rupture count matches CSV",
        "expected": export_counts.get(
            "ruptures"
        ),
        "actual": len(ruptures),
        "passes": (
            export_counts.get(
                "ruptures"
            )
            == len(ruptures)
        ),
    },
    {
        "check": (
            "Java Cascadia count matches CSV"
        ),
        "expected": export_counts.get(
            "cascadia_interface_ruptures"
        ),
        "actual": len(cascadia),
        "passes": (
            export_counts.get(
                "cascadia_interface_ruptures"
            )
            == len(cascadia)
        ),
    },
    {
        "check": (
            "Java Oregon count matches CSV"
        ),
        "expected": export_counts.get(
            "oregon_intraslab_ruptures"
        ),
        "actual": len(oregon),
        "passes": (
            export_counts.get(
                "oregon_intraslab_ruptures"
            )
            == len(oregon)
        ),
    },
]


validation = pd.DataFrame(
    validation_records
)

validation.to_csv(
    validation_output_path,
    index=False,
)


print("\nRupture-rate summary:")
display(summary)

print("\nOregon source counts by model combination:")
display(
    oregon_source_counts.rename(
        "source_count"
    ).reset_index()
)

print("\nCell 6 validation:")
display(validation)


def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as file_object:
        for block in iter(
            lambda: file_object.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


run_metadata = {
    "model_tag": MODEL_TAG,
    "nshmp_haz_tag": NSHMP_HAZ_TAG,
    "nshmp_haz_commit": NSHMP_HAZ_COMMIT,
    "nshmp_lib_jar": next(
        (
            str(path)
            for path in runtime_paths
            if "nshmp-lib" in path.name.lower()
        ),
        None,
    ),
    "java_path": java_path,
    "java_maximum_heap": "4g",
    "classpath_entries": len(
        classpath_paths
    ),
    "model_directory": str(
        MODEL_DIR
    ),
    "exporter_class": (
        "RuptureRateExporter"
    ),
    "run_started_at_utc": (
        run_started.isoformat()
    ),
    "run_finished_at_utc": (
        run_finished.isoformat()
    ),
    "run_duration_seconds": (
        run_duration_seconds
    ),
    "return_code": run_result.returncode,
    "raw_output": str(
        raw_output_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "raw_output_sha256": file_sha256(
        raw_output_path
    ),
    "cascadia_output": str(
        cascadia_output_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "cascadia_output_sha256": file_sha256(
        cascadia_output_path
    ),
    "oregon_output": str(
        oregon_output_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "oregon_output_sha256": file_sha256(
        oregon_output_path
    ),
    "rupture_rows": len(
        ruptures
    ),
    "cascadia_rows": len(
        cascadia
    ),
    "oregon_rows": len(
        oregon
    ),
    "validation_passed": bool(
        validation["passes"].all()
    ),
    "summary_file": str(
        summary_output_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "validation_file": str(
        validation_output_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "run_log": str(
        run_log_path.relative_to(
            PROJECT_ROOT
        )
    ),
}


run_metadata_path.write_text(
    json.dumps(
        run_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


if not validation["passes"].all():
    failed_checks = validation.loc[
        ~validation["passes"],
        [
            "check",
            "expected",
            "actual",
        ],
    ]

    print("\nFailed checks:")
    display(failed_checks)

    raise RuntimeError(
        "Cell 6 validation failed. Review the failed checks "
        "before using the rupture rates."
    )


print("\nCELL 6 VALIDATION PASSED")

print(f"\nTotal rupture rows:       {len(ruptures):,}")
print(f"Cascadia rupture rows:    {len(cascadia):,}")
print(f"Oregon rupture rows:      {len(oregon):,}")
print(
    "Weighted annual rate:    "
    f"{ruptures['weighted_annual_rate'].sum():.12g}"
)
print(
    "Run duration:            "
    f"{run_duration_seconds:,.2f} seconds"
)

print(f"\nRaw export:\n  {raw_output_path}")
print(f"\nCascadia rates:\n  {cascadia_output_path}")
print(f"\nOregon rates:\n  {oregon_output_path}")
print(f"\nSummary:\n  {summary_output_path}")
print(f"\nValidation:\n  {validation_output_path}")
print(f"\nRun metadata:\n  {run_metadata_path}")

print(
    "\nThe Cascadia interface and Oregon intraslab "
    "rupture-rate tables were exported and validated."
)

Running the rupture-rate exporter...
Model:  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\raw\usgs_nshm_conus_2018\nshm-conus-5.2.4\nshm-conus-5.2.4
Output: C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\interim\usgs_rupture_rate_exports\usgs_subduction_rupture_rates_raw.csv

The model-loading step may be quiet for several minutes.

Exporter output:
... 718 earlier lines were saved to the log file ...
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch

Reading the exported rupture table...

Rupture-rate summary:


,target_group,rupture_rows,source_trees,rupture_sets,sources,minimum_magnitude,maximum_magnitude,raw_rate_sum,weighted_rate_sum,unique_rakes,source_types
0,cascadia_interface,11101,1,21,21,8.00,9.34,0.017700,0.003307,1,INTERFACE
1,oregon_intraslab,12315,1,2,1642,6.55,7.95,0.001916,0.001916,1,SLAB



Oregon source counts by model combination:


,tree_id,tree_path,branch_index,rupture_set_id,source_count
0,8231,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/slab/OR,0,8211,821
1,8231,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/slab/OR,1,8212,821



Cell 6 validation:


,check,expected,actual,passes
0,Exporter completed successfully,True,True,True
1,Both target groups were exported,"[cascadia_interface, oregon_intraslab]","[cascadia_interface, oregon_intraslab]",True
2,Exported rupture table is not empty,> 0,23416,True
3,Rupture identifiers are complete,0,0,True
4,Rupture identifiers are unique,0,0,True
5,Core numeric values are finite,True,True,True
6,Raw annual rates are positive,True,True,True
7,Weighted annual rates are positive,True,True,True
8,"Rupture-set weights are in (0, 1]",True,True,True
9,Weighted rate equals raw rate times rupture-set weight,True,True,True



CELL 6 VALIDATION PASSED

Total rupture rows:       23,416
Cascadia rupture rows:    11,101
Oregon rupture rows:      12,315
Weighted annual rate:    0.00522322145746
Run duration:            98.16 seconds

Raw export:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\interim\usgs_rupture_rate_exports\usgs_subduction_rupture_rates_raw.csv

Cascadia rates:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\usgs_rupture_rates\cascadia_interface_rupture_rates.csv

Oregon rates:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\usgs_rupture_rates\oregon_intraslab_rupture_rates.csv

Summary:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\usgs_rupture_rate_summary.csv

Validation:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\usgs_rupture_rate_validation.csv

Run metadata:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance

In [ ]:
# from pathlib import Path

# import os
# import shutil
# import subprocess

# import pandas as pd
# from IPython.display import display


# runtime_manifest_path = (
#     METADATA_DIR
#     / "nshmp_haz_2_6_5_runtime_classpath.csv"
# )

# runtime_table = pd.read_csv(
#     runtime_manifest_path
# )

# runtime_paths = [
#     Path(path).resolve()
#     for path in runtime_table["path"].dropna()
#     if Path(path).exists()
# ]

# diagnostic_dir = (
#     JAVA_EXPORTER_DIR
#     / "tree_inventory"
# )

# diagnostic_dir.mkdir(
#     parents=True,
#     exist_ok=True,
# )

# java_source_path = (
#     diagnostic_dir
#     / "SourceTreeInventory.java"
# )

# classes_dir = (
#     diagnostic_dir
#     / "classes"
# )

# classes_dir.mkdir(
#     parents=True,
#     exist_ok=True,
# )

# output_path = (
#     METADATA_DIR
#     / "usgs_subduction_source_tree_inventory.csv"
# )

# compile_arguments_path = (
#     diagnostic_dir
#     / "javac_arguments.txt"
# )

# run_arguments_path = (
#     diagnostic_dir
#     / "java_arguments.txt"
# )


# java_source = r'''
# import gov.usgs.earthquake.nshmp.model.HazardModel;
# import gov.usgs.earthquake.nshmp.model.RuptureSet;
# import gov.usgs.earthquake.nshmp.model.Source;
# import gov.usgs.earthquake.nshmp.model.SourceTree;
# import gov.usgs.earthquake.nshmp.tree.Branch;

# import java.io.PrintWriter;
# import java.nio.charset.StandardCharsets;
# import java.nio.file.Files;
# import java.nio.file.Path;
# import java.util.Locale;

# public final class SourceTreeInventory {

#   private SourceTreeInventory() {}

#   public static void main(String[] args) throws Exception {

#     if (args.length != 2) {
#       throw new IllegalArgumentException(
#           "Expected model directory and output CSV.");
#     }

#     Path modelPath = Path.of(args[0]);
#     Path outputPath = Path.of(args[1]);

#     HazardModel model = HazardModel.load(modelPath);

#     if (outputPath.getParent() != null) {
#       Files.createDirectories(outputPath.getParent());
#     }

#     try (
#         PrintWriter writer = new PrintWriter(
#             Files.newBufferedWriter(
#                 outputPath,
#                 StandardCharsets.UTF_8))
#     ) {

#       writer.println(String.join(",",
#           "tree_id",
#           "tree_name",
#           "tree_path",
#           "tree_type",
#           "tectonic_setting",
#           "branch_index",
#           "branch_weight",
#           "rupture_set_id",
#           "rupture_set_name",
#           "rupture_set_type",
#           "rupture_set_weight",
#           "source_count"
#       ));

#       for (SourceTree tree : model) {

#         String treeType = tree.type().name();

#         if (
#             !treeType.contains("INTERFACE")
#             && !treeType.contains("SLAB")
#         ) {
#           continue;
#         }

#         for (
#             int branchIndex = 0;
#             branchIndex < tree.size();
#             branchIndex++
#         ) {

#           Branch<RuptureSet<? extends Source>> branch =
#               tree.get(branchIndex);

#           RuptureSet<? extends Source> ruptureSet =
#               branch.value();

#           writer.println(String.join(",",
#               integer(tree.id()),
#               csv(tree.name()),
#               csv(tree.path().toString().replace('\\', '/')),
#               csv(treeType),
#               csv(tree.setting().name()),
#               integer(branchIndex),
#               number(branch.weight()),
#               integer(ruptureSet.id()),
#               csv(ruptureSet.name()),
#               csv(ruptureSet.type().name()),
#               number(ruptureSet.weight()),
#               integer(ruptureSet.size())
#           ));
#         }
#       }
#     }

#     System.out.println("INVENTORY_COMPLETE");
#     System.out.println("output=" + outputPath);
#   }

#   private static String integer(int value) {
#     return Integer.toString(value);
#   }

#   private static String number(double value) {
#     return String.format(
#         Locale.US,
#         "%.17g",
#         value);
#   }

#   private static String csv(String value) {

#     if (value == null) {
#       return "";
#     }

#     boolean quote =
#         value.contains(",")
#         || value.contains("\"")
#         || value.contains("\n")
#         || value.contains("\r");

#     if (!quote) {
#       return value;
#     }

#     return "\""
#         + value.replace("\"", "\"\"")
#         + "\"";
#   }
# }
# '''.strip()


# java_source_path.write_text(
#     java_source + "\n",
#     encoding="utf-8",
# )


# classpath = os.pathsep.join(
#     str(path)
#     for path in runtime_paths
# )


# def argument_value(value):
#     text = str(value).replace(
#         "\\",
#         "\\\\",
#     )

#     text = text.replace(
#         '"',
#         '\\"',
#     )

#     return f'"{text}"'


# compile_arguments_path.write_text(
#     "\n".join(
#         [
#             "-encoding",
#             "UTF-8",
#             "-classpath",
#             argument_value(classpath),
#             "-d",
#             argument_value(classes_dir),
#             argument_value(java_source_path),
#         ]
#     )
#     + "\n",
#     encoding="utf-8",
# )


# javac_path = shutil.which("javac")

# compile_result = subprocess.run(
#     [
#         javac_path,
#         f"@{compile_arguments_path}",
#     ],
#     cwd=str(diagnostic_dir),
#     capture_output=True,
#     text=True,
#     encoding="utf-8",
#     errors="replace",
#     check=False,
# )


# if compile_result.returncode != 0:
#     print(compile_result.stdout)
#     print(compile_result.stderr)

#     raise RuntimeError(
#         "The source-tree inventory program did not compile."
#     )


# run_classpath = os.pathsep.join(
#     [
#         str(classes_dir),
#         *[
#             str(path)
#             for path in runtime_paths
#         ],
#     ]
# )


# run_arguments_path.write_text(
#     "\n".join(
#         [
#             "-Xmx4g",
#             "-Dfile.encoding=UTF-8",
#             "-classpath",
#             argument_value(run_classpath),
#             "SourceTreeInventory",
#             argument_value(MODEL_DIR),
#             argument_value(output_path),
#         ]
#     )
#     + "\n",
#     encoding="utf-8",
# )


# java_path = shutil.which("java")

# run_result = subprocess.run(
#     [
#         java_path,
#         f"@{run_arguments_path}",
#     ],
#     cwd=str(diagnostic_dir),
#     capture_output=True,
#     text=True,
#     encoding="utf-8",
#     errors="replace",
#     timeout=7200,
#     check=False,
# )


# if run_result.returncode != 0:
#     print(run_result.stdout)
#     print(run_result.stderr)

#     raise RuntimeError(
#         "The source-tree inventory program failed."
#     )


# tree_inventory = pd.read_csv(
#     output_path
# )


# search_text = (
#     tree_inventory[
#         [
#             "tree_name",
#             "tree_path",
#             "tree_type",
#             "rupture_set_name",
#             "rupture_set_type",
#         ]
#     ]
#     .fillna("")
#     .astype(str)
#     .agg(
#         " ".join,
#         axis=1,
#     )
#     .str.lower()
# )


# candidate_mask = search_text.str.contains(
#     r"oregon|slab|intraslab|cascadia",
#     regex=True,
# )


# candidate_trees = tree_inventory.loc[
#     candidate_mask,
#     [
#         "tree_id",
#         "tree_name",
#         "tree_path",
#         "tree_type",
#         "branch_index",
#         "branch_weight",
#         "rupture_set_id",
#         "rupture_set_name",
#         "rupture_set_type",
#         "rupture_set_weight",
#         "source_count",
#     ],
# ].copy()


# pd.set_option(
#     "display.max_colwidth",
#     200,
# )

# pd.set_option(
#     "display.max_rows",
#     200,
# )


# print("Candidate Cascadia and Oregon source trees:")
# display(candidate_trees)

# print("\nRows by tree type:")
# display(
#     tree_inventory[
#         "tree_type"
#     ].value_counts(
#         dropna=False
#     )
# )

# print(f"\nInventory saved to:\n{output_path}")

Candidate Cascadia and Oregon source trees:


,tree_id,tree_name,tree_path,tree_type,branch_index,branch_weight,rupture_set_id,rupture_set_name,rupture_set_type,rupture_set_weight,source_count
0,3199,Cascadia Subduction Zone,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/interface/Cascadia,INTERFACE,0,0.1800,3150,"Cascadia (segmented, 1-2, bottom)",INTERFACE,0.1800,1
1,3199,Cascadia Subduction Zone,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/interface/Cascadia,INTERFACE,1,0.0464,3172,"Cascadia (unsegmented, 1-2-3-4, top)",INTERFACE,0.0464,1
2,3199,Cascadia Subduction Zone,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/interface/Cascadia,INTERFACE,2,0.1350,3160,"Cascadia (unsegmented, 1-2-3, bottom)",INTERFACE,0.1350,1
3,3199,Cascadia Subduction Zone,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/interface/Cascadia,INTERFACE,3,0.3000,3111,"Cascadia (segmented, 1, middle)",INTERFACE,0.3000,1
4,3199,Cascadia Subduction Zone,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/interface/Cascadia,INTERFACE,4,0.1200,3112,"Cascadia (segmented, 1, top)",INTERFACE,0.1200,1
5,3199,Cascadia Subduction Zone,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/interface/Cascadia,INTERFACE,5,0.3000,3161,"Cascadia (segmented, 1-2-3, middle)",INTERFACE,0.3000,1
6,3199,Cascadia Subduction Zone,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/interface/Cascadia,INTERFACE,6,0.3000,3170,"Cascadia (full, bottom)",INTERFACE,0.3000,1
7,3199,Cascadia Subduction Zone,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/interface/Cascadia,INTERFACE,7,0.1200,3152,"Cascadia (segmented, 1-2, top)",INTERFACE,0.1200,1
8,3199,Cascadia Subduction Zone,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/interface/Cascadia,INTERFACE,8,0.1800,3160,"Cascadia (segmented, 1-2-3, bottom)",INTERFACE,0.1800,1
9,3199,Cascadia Subduction Zone,C:/Users/USER/Documents/GitHub/seismic-correlation-insurance-loss/data/raw/usgs_nshm_conus_2018/nshm-conus-5.2.4/nshm-conus-5.2.4/subduction/interface/Cascadia,INTERFACE,9,0.1158,3171,"Cascadia (unsegmented, 1-2-3-4, middle)",INTERFACE,0.1158,1



Rows by tree type:


tree_type
INTERFACE    21
SLAB          5
Name: count, dtype: int64


Inventory saved to:
C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\usgs_subduction_source_tree_inventory.csv


In [17]:
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display


raw_rate_path = (
    RAW_EXPORT_DIR
    / "usgs_subduction_rupture_rates_raw.csv"
)

official_audit_path = (
    METADATA_DIR
    / "usgs_rupture_set_official_rate_audit.csv"
)

merged_audit_path = (
    METADATA_DIR
    / "usgs_rupture_set_rate_comparison.csv"
)

tree_summary_path = (
    METADATA_DIR
    / "usgs_rupture_rate_tree_summary.csv"
)

magnitude_summary_path = (
    METADATA_DIR
    / "usgs_rupture_rate_magnitude_summary.csv"
)

final_validation_path = (
    METADATA_DIR
    / "notebook_2_final_validation.csv"
)

completion_metadata_path = (
    METADATA_DIR
    / "notebook_2_completion_metadata.json"
)


for required_path in [
    raw_rate_path,
    official_audit_path,
]:
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Required file not found:\n{required_path}"
        )


ruptures = pd.read_csv(
    raw_rate_path,
    low_memory=False,
)

official = pd.read_csv(
    official_audit_path,
    low_memory=False,
)


def relative_model_path(values):
    normalized = (
        values
        .fillna("")
        .astype(str)
        .str.replace(
            "\\",
            "/",
            regex=False,
        )
    )

    relative = normalized.str.extract(
        r"(?i)(/subduction/.*)$",
        expand=False,
    )

    relative = relative.fillna(
        normalized
    )

    return relative.str.lower()


ruptures["tree_path_key"] = relative_model_path(
    ruptures["tree_path"]
)

official["tree_path_key"] = relative_model_path(
    official["tree_path"]
)


stable_key = [
    "target_group",
    "tree_path_key",
    "rupture_set_id",
    "rupture_set_name",
    "rupture_set_type",
]


official_key_is_unique = not official.duplicated(
    stable_key
).any()


if not official_key_is_unique:
    duplicate_official = official.loc[
        official.duplicated(
            stable_key,
            keep=False,
        ),
        stable_key,
    ]

    print("Duplicate official audit keys:")
    display(duplicate_official)

    raise RuntimeError(
        "The official audit does not have unique "
        "rupture-set identifiers."
    )


source_counts = (
    ruptures[
        [
            *stable_key,
            "source_index",
            "source_id",
        ]
    ]
    .drop_duplicates()
    .groupby(
        stable_key,
        dropna=False,
    )
    .size()
    .rename(
        "csv_source_count"
    )
    .reset_index()
)


csv_summary = (
    ruptures
    .groupby(
        stable_key,
        dropna=False,
    )
    .agg(
        csv_tree_id=(
            "tree_id",
            "first",
        ),
        csv_tree_name=(
            "tree_name",
            "first",
        ),
        csv_tree_type=(
            "tree_type",
            "first",
        ),
        csv_branch_index=(
            "branch_index",
            "first",
        ),
        csv_branch_weight=(
            "branch_weight",
            "first",
        ),
        csv_rupture_set_weight=(
            "rupture_set_weight",
            "first",
        ),
        csv_rupture_count=(
            "rupture_id",
            "size",
        ),
        csv_raw_rate_sum=(
            "raw_annual_rate",
            "sum",
        ),
        csv_weighted_rate_sum=(
            "weighted_annual_rate",
            "sum",
        ),
    )
    .reset_index()
)


csv_summary = csv_summary.merge(
    source_counts,
    on=stable_key,
    how="left",
    validate="one_to_one",
)


csv_key_is_unique = not csv_summary.duplicated(
    stable_key
).any()


official_summary = official.rename(
    columns={
        "tree_id": "official_tree_id",
        "tree_name": "official_tree_name",
        "tree_type": "official_tree_type",
        "branch_index": "official_branch_index",
        "branch_weight": "official_branch_weight",
        "rupture_set_weight": (
            "official_rupture_set_weight"
        ),
        "source_count": "official_source_count",
        "rupture_count": "official_rupture_count",
        "iterated_raw_rate_sum": (
            "official_iterated_raw_rate_sum"
        ),
        "total_mfd_rate_sum": (
            "official_total_mfd_rate_sum"
        ),
        "iterated_weighted_rate_sum": (
            "official_iterated_weighted_rate_sum"
        ),
        "weighted_total_mfd_rate_sum": (
            "official_weighted_total_mfd_rate_sum"
        ),
    }
)


comparison = official_summary.merge(
    csv_summary,
    on=stable_key,
    how="outer",
    indicator=True,
    validate="one_to_one",
)


comparison[
    "rupture_count_matches"
] = (
    comparison[
        "official_rupture_count"
    ]
    == comparison[
        "csv_rupture_count"
    ]
)


comparison[
    "source_count_matches"
] = (
    comparison[
        "official_source_count"
    ]
    == comparison[
        "csv_source_count"
    ]
)


comparison[
    "raw_rate_matches_csv"
] = np.isclose(
    comparison[
        "official_iterated_raw_rate_sum"
    ],
    comparison[
        "csv_raw_rate_sum"
    ],
    rtol=1e-10,
    atol=1e-15,
    equal_nan=False,
)


comparison[
    "weighted_rate_matches_csv"
] = np.isclose(
    comparison[
        "official_iterated_weighted_rate_sum"
    ],
    comparison[
        "csv_weighted_rate_sum"
    ],
    rtol=1e-10,
    atol=1e-15,
    equal_nan=False,
)


comparison[
    "official_branch_and_set_weights_match"
] = np.isclose(
    comparison[
        "official_branch_weight"
    ],
    comparison[
        "official_rupture_set_weight"
    ],
    rtol=1e-12,
    atol=1e-15,
    equal_nan=False,
)


comparison[
    "csv_branch_and_set_weights_match"
] = np.isclose(
    comparison[
        "csv_branch_weight"
    ],
    comparison[
        "csv_rupture_set_weight"
    ],
    rtol=1e-12,
    atol=1e-15,
    equal_nan=False,
)


comparison[
    "official_and_csv_weights_match"
] = np.isclose(
    comparison[
        "official_rupture_set_weight"
    ],
    comparison[
        "csv_rupture_set_weight"
    ],
    rtol=1e-12,
    atol=1e-15,
    equal_nan=False,
)


comparison[
    "deprecated_total_mfd_matches_iteration"
] = np.isclose(
    comparison[
        "official_total_mfd_rate_sum"
    ],
    comparison[
        "official_iterated_raw_rate_sum"
    ],
    rtol=1e-10,
    atol=1e-15,
    equal_nan=False,
)


comparison[
    "raw_rate_absolute_difference"
] = (
    comparison[
        "official_iterated_raw_rate_sum"
    ]
    - comparison[
        "csv_raw_rate_sum"
    ]
).abs()


comparison[
    "weighted_rate_absolute_difference"
] = (
    comparison[
        "official_iterated_weighted_rate_sum"
    ]
    - comparison[
        "csv_weighted_rate_sum"
    ]
).abs()


comparison.to_csv(
    merged_audit_path,
    index=False,
)


official_weight_summary = (
    official
    .groupby(
        "target_group",
        dropna=False,
    )
    .agg(
        official_loaded_rupture_sets=(
            "rupture_set_id",
            "size",
        ),
        official_source_count=(
            "source_count",
            "sum",
        ),
        official_rupture_count=(
            "rupture_count",
            "sum",
        ),
        official_rupture_set_weight_sum=(
            "rupture_set_weight",
            "sum",
        ),
        official_raw_rate_sum=(
            "iterated_raw_rate_sum",
            "sum",
        ),
        official_weighted_rate_sum=(
            "iterated_weighted_rate_sum",
            "sum",
        ),
    )
    .reset_index()
)


csv_set_weights = (
    ruptures[
        [
            *stable_key,
            "rupture_set_weight",
        ]
    ]
    .drop_duplicates(
        stable_key
    )
)


csv_weight_summary = (
    csv_set_weights
    .groupby(
        "target_group",
        dropna=False,
    )
    .agg(
        csv_loaded_rupture_sets=(
            "rupture_set_id",
            "size",
        ),
        csv_rupture_set_weight_sum=(
            "rupture_set_weight",
            "sum",
        ),
    )
    .reset_index()
)


weight_summary = official_weight_summary.merge(
    csv_weight_summary,
    on="target_group",
    how="outer",
    validate="one_to_one",
)


weight_summary[
    "loaded_rupture_set_counts_match"
] = (
    weight_summary[
        "official_loaded_rupture_sets"
    ]
    == weight_summary[
        "csv_loaded_rupture_sets"
    ]
)


weight_summary[
    "rupture_set_weight_sums_match"
] = np.isclose(
    weight_summary[
        "official_rupture_set_weight_sum"
    ],
    weight_summary[
        "csv_rupture_set_weight_sum"
    ],
    rtol=1e-12,
    atol=1e-15,
    equal_nan=False,
)


weight_summary.to_csv(
    tree_summary_path,
    index=False,
)


magnitude_data = ruptures.copy()

magnitude_data[
    "magnitude_bin"
] = magnitude_data[
    "magnitude"
].round(
    4
)


magnitude_summary = (
    magnitude_data
    .groupby(
        [
            "target_group",
            "magnitude_bin",
        ],
        dropna=False,
    )
    .agg(
        rupture_rows=(
            "rupture_id",
            "size",
        ),
        raw_annual_rate=(
            "raw_annual_rate",
            "sum",
        ),
        weighted_annual_rate=(
            "weighted_annual_rate",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "target_group",
            "magnitude_bin",
        ]
    )
)


magnitude_summary.to_csv(
    magnitude_summary_path,
    index=False,
)


all_sets_matched = comparison[
    "_merge"
].eq(
    "both"
).all()


total_csv_weighted_rate = ruptures[
    "weighted_annual_rate"
].sum()


total_official_weighted_rate = official[
    "iterated_weighted_rate_sum"
].sum()


total_rates_match = bool(
    np.isclose(
        total_csv_weighted_rate,
        total_official_weighted_rate,
        rtol=1e-10,
        atol=1e-15,
    )
)


deprecated_mfd_matches = comparison[
    "deprecated_total_mfd_matches_iteration"
].all()


validation = pd.DataFrame(
    [
        {
            "check": (
                "Official rupture-set keys are unique"
            ),
            "blocking": True,
            "passes": official_key_is_unique,
        },
        {
            "check": (
                "CSV rupture-set keys are unique"
            ),
            "blocking": True,
            "passes": csv_key_is_unique,
        },
        {
            "check": (
                "All rupture sets matched using stable keys"
            ),
            "blocking": True,
            "passes": all_sets_matched,
        },
        {
            "check": (
                "Cascadia loaded rupture sets equal 21"
            ),
            "blocking": True,
            "passes": (
                (
                    official[
                        "target_group"
                    ]
                    == "cascadia_interface"
                ).sum()
                == 21
            ),
        },
        {
            "check": (
                "Oregon loaded rupture sets equal 2"
            ),
            "blocking": True,
            "passes": (
                (
                    official[
                        "target_group"
                    ]
                    == "oregon_intraslab"
                ).sum()
                == 2
            ),
        },
        {
            "check": (
                "Official and CSV rupture counts match"
            ),
            "blocking": True,
            "passes": comparison[
                "rupture_count_matches"
            ].all(),
        },
        {
            "check": (
                "Official and CSV source counts match"
            ),
            "blocking": True,
            "passes": comparison[
                "source_count_matches"
            ].all(),
        },
        {
            "check": (
                "Independently enumerated raw rates "
                "match the CSV"
            ),
            "blocking": True,
            "passes": comparison[
                "raw_rate_matches_csv"
            ].all(),
        },
        {
            "check": (
                "Independently enumerated weighted rates "
                "match the CSV"
            ),
            "blocking": True,
            "passes": comparison[
                "weighted_rate_matches_csv"
            ].all(),
        },
        {
            "check": (
                "Official branch and rupture-set weights match"
            ),
            "blocking": True,
            "passes": comparison[
                "official_branch_and_set_weights_match"
            ].all(),
        },
        {
            "check": (
                "CSV branch and rupture-set weights match"
            ),
            "blocking": True,
            "passes": comparison[
                "csv_branch_and_set_weights_match"
            ].all(),
        },
        {
            "check": (
                "Official and CSV rupture-set weights match"
            ),
            "blocking": True,
            "passes": comparison[
                "official_and_csv_weights_match"
            ].all(),
        },
        {
        "check": (
            "Official and exported source-family "
            "weight totals match"
        ),
        "blocking": True,
        "passes": bool(
            weight_summary[
                "loaded_rupture_set_counts_match"
            ].all()
            and weight_summary[
                "rupture_set_weight_sums_match"
            ].all()
    ),
},
        {
            "check": (
                "Total weighted annual rates match"
            ),
            "blocking": True,
            "passes": total_rates_match,
        },
        {
            "check": (
                "Expected rupture-row count retained"
            ),
            "blocking": True,
            "passes": len(
                ruptures
            ) == 23416,
        },
        {
            "check": (
                "No duplicate rupture identifiers"
            ),
            "blocking": True,
            "passes": (
                ruptures[
                    "rupture_id"
                ].duplicated().sum()
                == 0
            ),
        },
        {
            "check": (
                "Deprecated totalMfd diagnostic "
                "matches direct iteration"
            ),
            "blocking": False,
            "passes": deprecated_mfd_matches,
        },
    ]
)


validation.to_csv(
    final_validation_path,
    index=False,
)


print("Stable rupture-set comparison:")
display(
    comparison[
        [
            "target_group",
            "tree_path_key",
            "rupture_set_id",
            "rupture_set_name",
            "_merge",
            "official_source_count",
            "csv_source_count",
            "official_rupture_count",
            "csv_rupture_count",
            "official_iterated_raw_rate_sum",
            "csv_raw_rate_sum",
            "official_iterated_weighted_rate_sum",
            "csv_weighted_rate_sum",
            "raw_rate_matches_csv",
            "weighted_rate_matches_csv",
        ]
    ]
)


print("\nSource-family weight summary:")
display(
    weight_summary
)


print("\nCorrected Notebook 2 validation:")
display(
    validation
)


mfd_mismatches = comparison.loc[
    ~comparison[
        "deprecated_total_mfd_matches_iteration"
    ],
    [
        "target_group",
        "tree_path_key",
        "rupture_set_id",
        "rupture_set_name",
        "official_iterated_raw_rate_sum",
        "official_total_mfd_rate_sum",
    ],
]


if not mfd_mismatches.empty:
    print(
        "\nDeprecated totalMfd diagnostic differences:"
    )
    display(
        mfd_mismatches
    )


blocking_failures = validation.loc[
    validation[
        "blocking"
    ]
    & ~validation[
        "passes"
    ],
    "check",
].tolist()


def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as file_object:
        for block in iter(
            lambda: file_object.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(
                block
            )

    return digest.hexdigest()


completion_metadata = {
    "notebook": (
        "02_extract_usgs_rupture_rates.ipynb"
    ),
    "model": (
        f"USGS NSHM CONUS {MODEL_EDITION}"
    ),
    "model_tag": MODEL_TAG,
    "nshmp_haz_tag": NSHMP_HAZ_TAG,
    "nshmp_haz_commit": NSHMP_HAZ_COMMIT,
    "nshmp_lib_version": "1.7.3",
    "rupture_rows": int(
        len(ruptures)
    ),
    "cascadia_rupture_rows": int(
        (
            ruptures[
                "target_group"
            ]
            == "cascadia_interface"
        ).sum()
    ),
    "oregon_rupture_rows": int(
        (
            ruptures[
                "target_group"
            ]
            == "oregon_intraslab"
        ).sum()
    ),
    "loaded_rupture_sets": int(
        len(official)
    ),
    "total_weighted_annual_rate": float(
        total_csv_weighted_rate
    ),
    "cascadia_rupture_set_weight_sum": float(
        weight_summary.loc[
            weight_summary[
                "target_group"
            ]
            == "cascadia_interface",
            "official_rupture_set_weight_sum",
        ].iloc[0]
    ),
    "oregon_rupture_set_weight_sum": float(
        weight_summary.loc[
            weight_summary[
                "target_group"
            ]
            == "oregon_intraslab",
            "official_rupture_set_weight_sum",
        ].iloc[0]
),
    "deprecated_total_mfd_diagnostic_passed": bool(
        deprecated_mfd_matches
    ),
    "raw_export": str(
        raw_rate_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "raw_export_sha256": file_sha256(
        raw_rate_path
    ),
    "cascadia_export": str(
        cascadia_output_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "cascadia_export_sha256": file_sha256(
        cascadia_output_path
    ),
    "oregon_export": str(
        oregon_output_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "oregon_export_sha256": file_sha256(
        oregon_output_path
    ),
    "rupture_set_comparison": str(
        merged_audit_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "source_family_summary": str(
        tree_summary_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "magnitude_summary": str(
        magnitude_summary_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "final_validation": str(
        final_validation_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "blocking_validation_passed": (
        len(blocking_failures) == 0
    ),
    "completed_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


completion_metadata_path.write_text(
    json.dumps(
        completion_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


if blocking_failures:
    raise RuntimeError(
        "Corrected Notebook 2 validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in blocking_failures
        )
    )


print("\nNOTEBOOK 2 VALIDATION COMPLETE")

print(f"\nRupture rows:             {len(ruptures):,}")
print(f"Loaded rupture sets:      {len(official):,}")
print(
    "Cascadia set-weight sum: "
    f"{weight_summary.loc[weight_summary['target_group'] == 'cascadia_interface', 'official_rupture_set_weight_sum'].iloc[0]:.12g}"
)

print(
    "Oregon set-weight sum:   "
    f"{weight_summary.loc[weight_summary['target_group'] == 'oregon_intraslab', 'official_rupture_set_weight_sum'].iloc[0]:.12g}"
)
print(
    "Weighted annual rate:    "
    f"{total_csv_weighted_rate:.14g}"
)

print(
    "\nThe independently enumerated USGS rates agree "
    "with the exported rupture tables."
)

print(
    "\nNext notebook:"
    "\n03_generate_annual_event_catalog.ipynb"
)

Stable rupture-set comparison:


,target_group,tree_path_key,rupture_set_id,rupture_set_name,_merge,official_source_count,csv_source_count,official_rupture_count,csv_rupture_count,official_iterated_raw_rate_sum,csv_raw_rate_sum,official_iterated_weighted_rate_sum,csv_weighted_rate_sum,raw_rate_matches_csv,weighted_rate_matches_csv
0,cascadia_interface,/subduction/interface/cascadia,3110,"Cascadia (segmented, 1, bottom)",both,1,1,3.0,3,0.000435,0.000435,0.000078,0.000078,True,True
1,cascadia_interface,/subduction/interface/cascadia,3111,"Cascadia (segmented, 1, middle)",both,1,1,3.0,3,0.000435,0.000435,0.000130,0.000130,True,True
2,cascadia_interface,/subduction/interface/cascadia,3112,"Cascadia (segmented, 1, top)",both,1,1,3.0,3,0.000435,0.000435,0.000052,0.000052,True,True
3,cascadia_interface,/subduction/interface/cascadia,3140,"Cascadia (segmented, 4, bottom)",both,1,1,3.0,3,0.001000,0.001000,0.000037,0.000038,True,True
4,cascadia_interface,/subduction/interface/cascadia,3141,"Cascadia (segmented, 4, middle)",both,1,1,3.0,3,0.001000,0.001000,0.000063,0.000063,True,True
5,cascadia_interface,/subduction/interface/cascadia,3142,"Cascadia (segmented, 4, top)",both,1,1,3.0,3,0.001000,0.001000,0.000025,0.000025,True,True
6,cascadia_interface,/subduction/interface/cascadia,3150,"Cascadia (segmented, 1-2, bottom)",both,1,1,3.0,3,0.000391,0.000391,0.000070,0.000070,True,True
7,cascadia_interface,/subduction/interface/cascadia,3151,"Cascadia (segmented, 1-2, middle)",both,1,1,3.0,3,0.000391,0.000391,0.000117,0.000117,True,True
8,cascadia_interface,/subduction/interface/cascadia,3152,"Cascadia (segmented, 1-2, top)",both,1,1,3.0,3,0.000391,0.000391,0.000047,0.000047,True,True
9,cascadia_interface,/subduction/interface/cascadia,3160,"Cascadia (segmented, 1-2-3, bottom)",both,1,1,3.0,3,0.000174,0.000174,0.000031,0.000031,True,True



Source-family weight summary:


,target_group,official_loaded_rupture_sets,official_source_count,official_rupture_count,official_rupture_set_weight_sum,official_raw_rate_sum,official_weighted_rate_sum,csv_loaded_rupture_sets,csv_rupture_set_weight_sum,loaded_rupture_set_counts_match,rupture_set_weight_sums_match
0,cascadia_interface,21,21,11101.0,3.6068,0.017700,0.003307,21,3.6068,True,True
1,oregon_intraslab,2,1642,12315.0,2.0000,0.001916,0.001916,2,2.0000,True,True



Corrected Notebook 2 validation:


,check,blocking,passes
0,Official rupture-set keys are unique,True,True
1,CSV rupture-set keys are unique,True,True
2,All rupture sets matched using stable keys,True,True
3,Cascadia loaded rupture sets equal 21,True,True
4,Oregon loaded rupture sets equal 2,True,True
5,Official and CSV rupture counts match,True,True
6,Official and CSV source counts match,True,True
7,Independently enumerated raw rates match the CSV,True,True
8,Independently enumerated weighted rates match the CSV,True,True
9,Official branch and rupture-set weights match,True,True



NOTEBOOK 2 VALIDATION COMPLETE

Rupture rows:             23,416
Loaded rupture sets:      23
Cascadia set-weight sum: 3.6068
Oregon set-weight sum:   2
Weighted annual rate:    0.0052232214574573

The independently enumerated USGS rates agree with the exported rupture tables.

Next notebook:
03_generate_annual_event_catalog.ipynb
